# Kharcha reader: build, train, test and export the models

One self-contained notebook. **Nothing is imported from `reader_server`**: every piece of the reader is written out in the cells below,
so you can read it, change it and run it. Run top to bottom on the free Colab CPU (about 10 to 15 minutes, mostly OCR).

| Part | What happens |
|---|---|
| 1 | Install docTR, Tesseract (English + Nepali) and the Python packages |
| 2 | **Settings**: the few numbers you may want to change (review threshold, how cautious the category model is) |
| 3 | **Reading rules** (`parse`): turn OCR text lines into merchant, date, total, items; the arithmetic check |
| 4 | **OCR**: docTR, with Tesseract as the rescue path; the "is this even readable?" check |
| 5 | **The two trained models**: the line model (what is each line?) and the category model (what is each item?) |
| 6 | **Training data**: fake receipts plus your 25 real bills, and how to train |
| 7 | **Test 1**: the models on fake receipts they have never seen |
| 8 | **Test 2**: the models on real bills they have never seen (the honest test), then a PASS / FAIL list |
| 9 | **Train the final models**, check the hard bills |
| 10 | **Your own photo**: any image in, full explanation + JSON out |
| 11 | **Export**: a folder with the models and one `kharcha_reader.py` for the backend, tested in a fresh process |

**What changed in this version (the problems found on bills 1, 3 and 21)**
* `HSC:` (the HS-code column of Bhat-Bhateni bills) is no longer read as an item, and an item whose name and numbers the OCR put on two lines is joined back together.
* The **Grand Total** line now wins over a plain "Total" line, and `4802/50` is read as 4802.50. A total that is exactly items + 13% VAT is accepted even when the VAT line itself is unreadable.
* The date is searched in the "Date" lines as well, so a garbled "Bill #" line can no longer hide the real date. If a date was printed but misread (`2082-47-78`) it says so instead of just "not found".
* The category model says **Other** unless it is clearly sure (a 0.39 guess of "Dining" for a chair is now "Other").
* If the line model finds no item at all but a total exists, the plain rules look for a "name  amount" line (this is what rescues a one-line handwritten bill).
* Text that is just noise (handwritten Nepali read as Latin) is reported as **not readable**, with no invented items.

## 1. Install

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-nep > /dev/null
!pip -q install "python-doctr[torch]" pytesseract opencv-python-headless scikit-learn joblib scipy pillow numpy
!tesseract --list-langs

## 2. Settings
Everything you may want to tune is in this one cell. The cells below read these names.
(Cells that start with `# export` are the ones copied into `kharcha_reader.py` in part 11. Do not delete that first line.)

In [ ]:
# export order=0 name=settings
# Every number you may want to change is here. The cells below read these names.
import os, re, io, json, math, random, statistics, unicodedata, zipfile, glob, shutil, time
from datetime import date as _date, timedelta

REVIEW_THRESHOLD = 0.7        # confidence at or above this: filed automatically; below: the review screen
CATEGORY_MIN_PROB = 0.45      # an item whose best category is less likely than this becomes "Other" (the user decides)
CATEGORY_MIN_MARGIN = 0.10    # ... or whose best category is not at least this much ahead of the second best
RESCUE = 0.25                 # a "name  amount" line the line model only half believes is an item (p >= this) is kept as one
os.environ.setdefault("OCR_ENGINE", "auto")          # auto = docTR, with Tesseract (English + Nepali) as the rescue path
os.environ.setdefault("KHARCHA_DATE_ORDER", "mdy")   # how to read an ambiguous 05/10/2026: mdy (Nepali POS printers) or dmy
os.environ.setdefault("USE_TORCH", "1")              # Colab also has TensorFlow; make docTR use PyTorch
os.environ.setdefault("USE_TF", "0")
print("settings loaded")

## 3. Reading rules (`parse`)
Plain rules, no model. They pull the numbers out of the lines the trained model picked: the amount at the end of a line, the quantity, the date (including Nepali Bikram Sambat dates, converted to an approximate Gregorian date), and the **arithmetic check**
(do the items plus VAT minus discount add up to the total?). That check is the strongest signal that a read is right, so it drives the confidence.

In [ ]:
# export order=1 name=parse
import os
import re
import unicodedata
from datetime import date as _date, timedelta

_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
_NUM = r"\d[\d,]*(?:\.\d{1,2})?"
_NUM_AT_END = re.compile(rf"(?<![\w.,/-])({_NUM})\s*(?:/-|-)?\s*$")
_ANY_NUM = re.compile(rf"(?<![\w.,/-])({_NUM})(?![\w/-]*\d)")
_CURRENCY = re.compile(r"(?i)\b(?:rs|npr|inr|usd)\b\.?|रु\.?|₹|\$")

# Lines containing these are never items.
_NOT_ITEM = re.compile(
    r"(?i)sub\s*-?\s*total|\b(?:total|grand|gross|tax(?:able)?|vat|service|discount|change|cash|tender|paid|balance|amount|"
    r"round(?:ing|ed)?|card|visa|qr|tip|invoice|table|date|earned|points?|savings?|qty|cashier|counter)s?\b|\btime\s*:|bill\s*no|\bpan\b|\bphone\b|\btel\b|"
    r"कुल|जम्मा|उप\s*जम्मा|कर|छुट|फिर्ता|नगद|भ्याट|सेवा|रकम|मिति|बिल\s*नं|प्यान"
)      # whole words only: "Vegetable", "Cashew", "Cardamom" and "Time Pass" are items, "Table 4" and "Cash" are not
_TOTAL_WORDS = re.compile(r"(?i)grand\s*total|net\s*(?:amount|total|payable)|amount\s*(?:due|payable)|total|कुल|जम्मा")
_NOT_TOTAL = re.compile(r"(?i)sub\s*-?\s*total|tax|vat|before|excl|उप\s*जम्मा|कर|भ्याट")
_STRONG_TOTAL = re.compile(r"(?i)grand|net|payable|due|जम्मा")
_NOISE_TOP = re.compile(
    r"(?i)invoice|bill|receipt|tax|vat|pan\b|reg|phone|tel\b|address|date|welcome|thank|www|http|@|"
    r"cash\s*memo|बिल|रसिद|प्यान|मिति|कर"
)
_HSC = re.compile(r"(?i)\bHSC\b\s*:?|\bHS\s*code\b\s*:?")      # the "HS code" column on Bhat-Bhateni style bills
_MONTHS = {m: i + 1 for i, m in enumerate("jan feb mar apr may jun jul aug sep oct nov dec".split())}


def unwrap_json(o):
    """Decode JSON stored inside JSON strings (how the v2 dataset keeps its labels)."""
    import json
    if isinstance(o, str):
        t = o.strip()
        if t[:1] in "{[":
            try:
                return unwrap_json(json.loads(t))
            except Exception:
                return o
        return o
    if isinstance(o, dict):
        return {k: unwrap_json(v) for k, v in o.items()}
    if isinstance(o, list):
        return [unwrap_json(v) for v in o]
    return o


_LOOKALIKE = str.maketrans({"o": "0", "O": "0", "l": "1", "I": "1", "'": "", "\u2019": "", "`": ""})


def normalize(text: str) -> str:
    t = text.translate(_DIGITS)
    t = re.sub(r"(?<![/\-\d])(\d{3,})/(\d{2})\s*$", r"\1.\2", t)               # OCR writes "4802/50" for 4802.50
    t = re.sub(r"(?<=\d)[ \t]*([.,])[ \t]*(?=\d)", r"\1", t)    # OCR often leaves "125 .00" or "120. 00"
    t = re.sub(r"(?<=\d)\.{2,}(?=\d)", ".", t)
    t = re.sub(r"(?<=\d),(\d{2})\s*$", r".\1", t)                  # "309,73" at the end of a line is 309.73 (not a thousands group)
    m = re.search(r"(\S+)\s*$", t)                                # "960.0o" -> "960.00" (look-alike letters in the amount)
    if m and re.search(r"\d", m.group(1)) and re.fullmatch(r"[\dolOI.,'\u2019`]+", m.group(1)):
        t = t[: m.start(1)] + m.group(1).translate(_LOOKALIKE)
    return t


def to_amount(s):
    """'1,234.50' -> 1234.5 (None if it is not a plain amount)."""
    if s is None:
        return None
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return None


def _has_letter(s: str) -> bool:
    return any(ch.isalpha() for ch in s)


def _last_amount(line: str):
    line = _CURRENCY.sub(" ", normalize(line)).strip()
    m = _NUM_AT_END.search(line)
    if m:
        return to_amount(m.group(1)), line[: m.start()].strip()
    return None, line


def find_total(lines):
    cands = []
    for i, raw in enumerate(lines):
        line = normalize(raw)
        if _TOTAL_WORDS.search(line) and not _NOT_TOTAL.search(line):
            amt, _ = _last_amount(line)
            if amt is None and i + 1 < len(lines):          # amount sits on the next line
                amt, _ = _last_amount(lines[i + 1])
            if amt is not None and amt > 0:
                cands.append((bool(_STRONG_TOTAL.search(line)), i, amt))
    if cands:
        strong = [c for c in cands if c[0]]
        return (strong or cands)[-1][2]
    return None


def find_date(text: str):
    t = normalize(text)
    m = re.search(r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})", t)
    if m:
        y, mo, d = int(m[1]), int(m[2]), int(m[3])
        if 2050 <= y <= 2100:        # Bikram Sambat year: not converted, so no guess
            return None
        return _valid(y, mo, d)
    m = re.search(r"(\d{1,2})[-/.](\d{1,2})[-/.](\d{4})\b", t)
    if m:
        a, b, y = int(m[1]), int(m[2]), int(m[3])
        if y >= 2050:
            return None
        day, mon = (b, a) if b > 12 >= a else (a, b)
        return _valid(y, mon, day)
    m = re.search(r"(\d{1,2})[-/.](\d{1,2})[-/.](\d{2})\b", t)
    if m:
        a, b, y = int(m[1]), int(m[2]), 2000 + int(m[3])
        day, mon = (b, a) if b > 12 >= a else (a, b)
        return _valid(y, mon, day)
    m = re.search(r"(?i)(\d{1,2})[ \-.,]*([a-z]{3})[a-z]*[ \-.,]*(\d{4})", t)
    if m and m[2].lower() in _MONTHS:
        return _valid(int(m[3]), _MONTHS[m[2].lower()], int(m[1]))
    return None


def _valid(y, mo, d):
    try:
        return _date(y, mo, d).isoformat()
    except ValueError:
        return None


# --- Bikram Sambat (Nepali calendar) -------------------------------------------------------------
# Month start dates (Gregorian month, day) for BS months 1..12 (Baisakh..Chaitra). They move by a day or two
# between years, so a converted date is approximate (about +-2 days): good enough to pick the right month.
DATE_ORDER = os.environ.get("KHARCHA_DATE_ORDER", "mdy").lower()      # which way to read an ambiguous 05/10/2026
_BS_STARTS = [(4, 14), (5, 15), (6, 15), (7, 17), (8, 17), (9, 17), (10, 18), (11, 17), (12, 16), (1, 14), (2, 13), (3, 14)]


def bs_to_ad(y, mo, d):
    """Approximate Bikram Sambat -> Gregorian ISO date, or None if it is not a plausible BS date."""
    if not (2070 <= y <= 2100 and 1 <= mo <= 12 and 1 <= d <= 32):
        return None
    gm, gd = _BS_STARTS[mo - 1]
    gy = y - 57 + (1 if mo >= 10 else 0)             # BS year starts mid-April of (y - 57)
    try:
        return (_date(gy, gm, gd) + timedelta(days=d - 1)).isoformat()
    except ValueError:
        return None


def find_date_bs(text: str):
    """A Bikram Sambat date in the text (year 207x-209x, y/m/d or d/m/y), converted to an approximate Gregorian date."""
    t = normalize(text)
    m = re.search(r"\b(20[7-9]\d)[-/.](\d{1,2})[-/.](\d{1,2})\b", t)
    if m:
        return bs_to_ad(int(m[1]), int(m[2]), int(m[3]))
    m = re.search(r"\b(\d{1,2})[-/.](\d{1,2})[-/.](20[7-9]\d)\b", t)
    if m:
        return bs_to_ad(int(m[3]), int(m[2]), int(m[1]))
    return None


def _plausible(d, today):
    return bool(d) and _date(2015, 1, 1) <= _date.fromisoformat(d) <= today + timedelta(days=2)


def resolve_date(text: str, today=None):
    """The receipt's date, or None. A date in the future or before 2015 is an OCR misread ("2036"), so it is dropped
    (the Bikram Sambat date is tried instead, else the user types it)."""
    today = today or _date.today()
    d = _resolve_date_raw(text, today)
    if d and not _plausible(d, today):
        bs = find_date_bs(normalize(text))
        return bs if _plausible(bs, today) else None
    return d


def resolve_date_info(text: str, today=None):
    """(date, approx): approx is True when the date was converted from a Bikram Sambat date (about +-2 days).
    Tries the whole text first, then the lines that say "date", then every line, so a garbled "Bill #" line that holds
    a future date no longer hides the real "Transaction Date" line."""
    today = today or _date.today()

    def one(t):
        d = resolve_date(t, today)
        return d, bool(d) and d == find_date_bs(normalize(t))
    lines = [l for l in text.split("\n") if l.strip()]
    labelled = [l for l in lines if re.search(r"(?i)date|\u092e\u093f\u0924\u093f", l) and not re.search(r"(?i)bill\s*(?:#|no)", l)]
    for t in [text] + labelled + lines:
        d, ap = one(t)
        if d:
            return d, ap
    return None, False


def _resolve_date_raw(text: str, today=None):
    """The receipt's date. Tries the plain Gregorian reading first. When day and month could be swapped
    (05/10/2026), a Bikram Sambat date on the same receipt decides, then 'not in the future', then day-first.
    If only a BS date exists it is converted (approximately)."""
    today = today or _date.today()
    t = normalize(text)
    bs = find_date_bs(t)
    m = re.search(r"(\d{1,2})[-/.](\d{1,2})[-/.](20[0-4]\d)\b", t)
    if m:
        a, b, y = int(m[1]), int(m[2]), int(m[3])
        if a != b and a <= 12 and b <= 12:
            day_first, month_first = _valid(y, b, a), _valid(y, a, b)
            # The Nepali POS printers on the sample bills print MM/DD/YYYY (confirmed by their Bikram Sambat dates),
            # so month-first is the default; KHARCHA_DATE_ORDER=dmy flips it.
            cands = [c for c in ((month_first, day_first) if DATE_ORDER == "mdy" else (day_first, month_first)) if c]
            if len(cands) == 2:
                if bs:
                    ref = _date.fromisoformat(bs)
                    return min(cands, key=lambda c: abs((_date.fromisoformat(c) - ref).days))
                ok = [c for c in cands if _date.fromisoformat(c) <= today]
                if len(ok) == 1:
                    return ok[0]
                return cands[0]
    return find_date(t) or bs


_ADD = re.compile(r"(?i)\bvat\b|\btax\b|service\s*(?:charge|tax)|कर|भ्याट|सेवा")
_SUB = re.compile(r"(?i)discount|\bdisc\b|छुट")
_CASH = re.compile(r"(?i)\bcash\b|tender|paid|received|नगद")
_CHANGE = re.compile(r"(?i)change|refund|return|फिर्ता")


def arithmetic_check(lines, items, total, detail=None):
    """Does the arithmetic of the receipt hold? Returns "items" when item amounts (+ VAT / service charge - discount
    printed on the bill) add up to the total, "paid" when only cash minus change equals the total, else None.
    A wrong or half-read receipt rarely adds up by accident, so this is a better confidence signal than the OCR's own."""
    if not total or total <= 0:
        return None
    tol = max(0.5, 0.0003 * total)          # bills round to the rupee at most; one misread digit must not pass
    adds, subs, cash, change = [0.0], [0.0], [], []
    for raw in lines:
        a = line_amount(raw)
        if a is None or a <= 0:
            continue
        t = normalize(raw)
        if re.search(r"(?i)\b(?:vat|pan)\s*(?:no|number|#)", t) or a > total * 2:       # "VAT No: 606762403" is not a tax amount
            continue
        if _SUB.search(t):
            subs.append(a)
        elif _ADD.search(t):
            adds.append(a)
        elif _CASH.search(t):
            cash.append(a)
        elif _CHANGE.search(t):
            change.append(a)
    base = sum(i["price"] for i in items)
    if detail is not None:
        detail.update(items_sum=round(base, 2), tax_amounts_found=[a for a in adds if a], discount_amounts_found=[a for a in subs if a],
                      cash_found=cash, change_found=change, tolerance=round(tol, 2), printed_total=total)
    if items:
        for v in adds:
            for w in adds:
                for d in subs:
                    if abs(base + v + (w if w != v else 0.0) - d - total) <= tol:
                        if detail is not None:
                            detail.update(matched=f"items {base:.2f} + tax {v + (w if w != v else 0.0):.2f} - discount {d:.2f} = {base + v + (w if w != v else 0.0) - d:.2f} = total")
                        return "items"
    if items:
        tol2 = max(1.0, 0.001 * total)
        for mult, why in ((1.13, "13% VAT"), (1.10 * 1.13, "10% service charge + 13% VAT")):
            if abs(base * mult - total) <= tol2:
                if detail is not None:
                    detail.update(matched=f"items {base:.2f} + {why} (the tax line itself was unreadable, rate inferred) = {base * mult:.2f} = total")
                return "items"
    if any(abs(c - total) <= tol for c in cash) or any(abs(c - h - total) <= tol for c in cash for h in change):
        if detail is not None:
            detail.update(matched="cash paid minus change equals the total (items not confirmed)")
        return "paid"
    return None


def find_merchant(lines):
    for raw in lines[:6]:
        line = re.sub(r"^[\W_]+|[\W_]+$", "", raw.strip())
        if not (3 <= len(line) <= 40) or _NOISE_TOP.search(line):
            continue
        # Devanagari vowel signs are marks (category M), not letters, so count both.
        letters = sum(unicodedata.category(ch)[0] in "LM" for ch in line)
        if letters / len(line) >= 0.6:
            return line
    return None


def parse_item_line(raw, strict=True, serial=False):
    """One OCR line -> {name, qty, price} or None. strict=True also rejects lines that
    look like totals, tax, dates and so on (the rules path); the trained line model
    makes that decision itself, so it calls this with strict=False.
    serial=True: the receipt numbers its lines (1, 2, 3 ...), so a leading number is a serial, not a quantity."""
    line = _HSC.sub(" ", _CURRENCY.sub(" ", normalize(raw))).strip()
    line = re.sub(r"(?<=\s)[=|~:;¢_]+(?=\s|$)", " ", line)         # stray OCR marks between the columns ("1 341.00 = 341.00")
    line = re.sub(r"\s{2,}", " ", line).strip()
    if not line:
        return None
    if strict:
        if _NOT_ITEM.search(line) or re.search(r"\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}", line):
            return None
    price, rest = _last_amount(line)
    if price is None or price <= 0 or price >= 10_000_000 or not _has_letter(rest):
        return None
    qty = 1
    if serial:
        rest = re.sub(r"^\d{1,3}[.)]?\s+(?=\S)", "", rest)
    tail = re.search(rf"(?:^|\s)(\d{{1,3}})\s+({_NUM})\s*$", rest)          # "... 4 400.00" before the amount 1600.00
    if tail and abs(int(tail[1]) * (to_amount(tail[2]) or 0) - price) <= 0.5:
        qty = int(tail[1])
        rest = re.sub(r"^\d{1,3}[.)]?\s+(?=\D)", "", rest[: tail.start()].strip())    # also drop a leading serial number "1 "
        name = re.sub(r"^[\W_]+|[\W_]+$", "", rest).strip()
        if len(name) < 2 or not _has_letter(name):
            return None
        return {"name": name, "qty": max(qty, 1), "price": price}
    m = re.search(r"(?:^|\s)(\d{1,3})\s*[xX×*@]{1,2}\s*\d[\d,]*(?:\.\d{1,2})?", rest)
    if m:
        qty = int(m[1])
        rest = (rest[: m.start()] + " " + rest[m.end():]).strip()
    else:
        m = re.match(r"^(\d{1,3})\s*[xX×*]{1,2}\s+(?=\D)", rest) or (None if serial else re.match(r"^(\d{1,3})\s+(?=\D)", rest))
        if m:
            qty = int(m[1])
            rest = rest[m.end():]
    rest = re.sub(rf"(?:\s+{_NUM})+\s*$", "", rest)          # drop unit prices left of the line total
    name = re.sub(r"^[\W_]+|[\W_]+$", "", rest).strip()
    if len(name) < 2 or not _has_letter(name):
        return None
    return {"name": name, "qty": max(qty, 1), "price": price}


def numbered(lines):
    """True when most of these lines start with 1, 2, 3 ... (a serial-number column, as on Bhat-Bhateni bills)."""
    lead = [int(m[1]) for m in (re.match(r"^\W*(\d{1,3})[.)]?\s+\D", normalize(l)) for l in lines) if m]
    if len(lead) < 3:
        return False
    rising = sum(1 for a, b in zip(lead, lead[1:]) if 0 < b - a <= 5)      # gaps happen when the OCR drops a line
    return rising >= 0.8 * (len(lead) - 1)


def find_items(lines):
    return [i for i in (parse_item_line(l) for l in lines) if i]


def line_amount(raw):
    """The amount at the end of a line, or None."""
    return _last_amount(raw)[0]


_NAMELESS = re.compile(r"^\W*(?:HSC|HS\s*code)\W*:?\s*((?:\d[\d,]*(?:\.\d{1,2})?\s*){1,3})$", re.I)


def join_split_items(lines, joined=None):
    """On some bills the OCR puts an item's numbers (\"HSC: 45.00 45.00\") on one line and its name (\"3 2PM AKABARE\") on the next.
    Put them back together. joined: pass a list to be told which lines were merged."""
    out, i = [], 0
    while i < len(lines):
        m = _NAMELESS.match(normalize(lines[i]))
        if m and i + 1 < len(lines):
            nxt = lines[i + 1]
            if _has_letter(nxt) and line_amount(nxt) is None and not _NOT_ITEM.search(normalize(nxt)):
                out.append(nxt.strip() + " " + m.group(1).strip())
                if joined is not None:
                    joined.append((lines[i], nxt))
                i += 2
                continue
        out.append(lines[i])
        i += 1
    return out


def looks_like_receipt(lines):
    """False when the text is noise (handwriting or a script the OCR cannot read, e.g. Devanagari read as Latin):
    not one total / tax / date / bill word, and fewer than 3 lines that end in an amount."""
    kw = sum(bool(_NOT_ITEM.search(normalize(l)) or re.search(r"(?i)\brs\b|\u0930\u0941|\brate\b", l)) for l in lines)
    prices = sum(line_amount(l) is not None and 0 < line_amount(l) < 1_000_000 and _has_letter(l) for l in lines)      # a 7-digit number is an ID or phone, not a price
    return kw >= 1 or prices >= 3


def parse_receipt(lines, line_model=None, trace=None):
    """lines: text lines, top to bottom.

    With a trained line model (line_model.LineModel) the MODEL decides which line is the
    merchant, an item or the total; the rules above only pull the numbers out of the lines
    it picked. Without one, the rules decide everything (the baseline to beat).
    """
    lines = [l for l in (s.strip() for s in lines) if l]
    joined = []
    lines = join_split_items(lines, joined)
    if trace is not None:
        trace["rejoined_lines"] = joined
    if line_model is None:
        return _parse_rules(lines)
    return _parse_model(lines, line_model, trace)


def _parse_rules(lines):
    items = find_items(lines)
    total = find_total(lines)
    if total is None and items:
        total = round(sum(i["price"] for i in items), 2)
    out = {"total": total, "items": items}
    merchant = find_merchant(lines)
    if merchant:
        out["merchant"] = merchant
    d, approx = resolve_date_info("\n".join(lines))
    if d:
        out["date"] = d
        if approx:
            out["date_approx"] = True
    return out


RESCUE = 0.25


def _parse_model(lines, model, trace=None):
    if not lines:
        return {"total": None, "items": [], "confidence": 0.0}
    probs = model.predict_proba(lines)               # list of {label: p} per line
    labels = [max(p, key=p.get) for p in probs]
    roles = [""] * len(lines)
    items, item_p = [], []
    for k, (line, lab, p) in enumerate(zip(lines, labels, probs)):
        # Rescue: a plain "name  amount" line the model only half-believes is an item (p >= RESCUE) is kept as one,
        # unless it holds a total/tax/cash/date word or the model is fairly sure it is something else.
        noise = bool(_NOT_ITEM.search(normalize(line)))
        rescue = lab == "other" and p["item"] >= RESCUE and p["other"] < 0.6 and not noise
        # A line holding a total / tax / cash / date word is never an item, whatever the model says (whole words only).
        if (lab == "item" and not noise) or rescue:
            it = parse_item_line(line, strict=False)
            if it:
                items.append(it)
                item_p.append(p["item"])
                roles[k] = "ITEM (rescued: model was unsure)" if rescue else "ITEM"
            else:
                roles[k] = "model said item, but no price found at the end of the line"
        elif lab == "item" and noise:
            roles[k] = "model said item, rejected: line has a total/tax/cash/date word"
    used = [k for k, r in enumerate(roles) if r.startswith("ITEM")]
    if numbered([lines[k] for k in used]):          # the bill numbers its lines: a leading number is a serial, not a quantity
        items = [parse_item_line(lines[k], strict=False, serial=True) or it for k, it in zip(used, items)]
    total, total_p, from_line, total_at, total_src = None, 0.0, False, None, "none"
    cands = []
    for i, (line, lab, p) in enumerate(zip(lines, labels, probs)):
        if lab == "total" and not _NOT_TOTAL.search(normalize(line)):        # never a subtotal, tax or "taxable amount" line
            amt = line_amount(line)
            if amt is None and i + 1 < len(lines):
                amt = line_amount(lines[i + 1])
            if amt is not None and amt > 0:
                cands.append((i, amt, p["total"], bool(_STRONG_TOTAL.search(normalize(line)))))
    for i, amt, pt, _ in ([c for c in cands if c[3]] or cands):               # a "Grand Total" / "Net" line beats a plain "Total" line
        if pt >= total_p - 0.15:
            total, total_p, from_line, total_at = amt, max(total_p, pt), True, i
    if total is not None:
        total_src = "the model's total line"
    if total is None:
        total = find_total(lines)                    # the model found no total line: ask the rules
        total_p, from_line = (0.3, True) if total is not None else (0.0, False)
        if total is not None:
            total_src = "rules (the model found no total line)"
    if total is None and items:
        total, total_p = round(sum(i["price"] for i in items), 2), 0.2
        total_src = "estimated: sum of the items (no total line found)"
    if not items and total is not None:     # the model found no item line at all: ask the plain rules (a name and an amount, nothing like a total or date)
        for k, line in enumerate(lines):
            if labels[k] not in ('item', 'other') or roles[k]:
                continue
            it = parse_item_line(line)
            if it and it['price'] <= total and len(re.sub(r'[^A-Za-z\u0900-\u097F]', '', it['name'])) >= 3:
                items.append(it)
                item_p.append(0.3)
                roles[k] = 'ITEM (rules: the model found no item line)'
    out = {"total": total, "items": items}
    top = [(p["merchant"], l) for l, p in zip(lines[:8], probs[:8])]
    merchant_at = None
    if top and max(top)[0] >= 0.4:
        m = re.sub(r"^[\W_]+|[\W_]+$", "", max(top)[1]).strip()
        if m:
            out["merchant"] = m
            merchant_at = top.index(max(top))
    d, approx = resolve_date_info("\n".join(lines))
    if d:
        out["date"] = d
        if approx:
            out["date_approx"] = True
    date_note = None
    if not d:
        m = next((l for l in lines if re.search(r"(?i)date|\u092e\u093f\u0924\u093f", l) and re.search(r"\d{1,4}\s*[-/.]\s*\d{1,2}\s*[-/.]\s*\d{1,4}", normalize(l))), None)
        if m:
            date_note = f"a date was printed ({m.strip()[:50]!r}) but what the OCR read is not a valid date (a misread digit); type it in or use the upload day"
    # Confidence: how sure the model was about the two things the app depends on.
    mean_p = sum(item_p) / len(item_p) if item_p else 0.0
    conf = min(total_p, mean_p)
    check, detail, rule = None, {}, "no total line found, so the model's own sureness is used"
    conf_model = conf
    if from_line:
        # Arithmetic is the stronger signal: if the amounts add up the read is almost surely right; a total that
        # a line was found for but that the items do not explain caps the confidence (review screen).
        check = arithmetic_check(lines, items, total, detail)
        if check == "items":
            conf = max(conf, 0.75 + 0.2 * min(total_p, max(mean_p, 0.0)))
            rule = "items add up to the total: confidence raised to at least 0.75"
        elif check == "paid":
            conf = min(max(conf, 0.6), 0.65)       # the total is corroborated but the items are not: always review
            rule = "only cash minus change matches the total: capped at 0.65 (always review)"
        else:
            conf = min(conf, 0.5)
            rule = "items do not add up to the total: capped at 0.5 (always review)"
    out["confidence"] = round(min(conf, 0.95), 3)
    out["needs_review"] = out["confidence"] < REVIEW_THRESHOLD or total is None
    out["reconciliation"] = (
        "No total found" if total is None else
        "Items add up to the total (VAT and discount included)" if check == "items" else
        "Total matches cash minus change; some items may be missed" if check == "paid" else
        "Items do not add up to the total" if from_line else
        "No total line found; total estimated from items")
    if trace is not None:
        date_at = next((i for i, l in enumerate(lines) if resolve_date_info(l)[0]), None)
        trace.update(
            lines=[{"index": i, "text": l, "model_label": lab, "model_sureness": round(p[lab], 2),
                    "top3": {k: round(v, 2) for k, v in sorted(p.items(), key=lambda kv: -kv[1])[:3]},
                    "role": roles[i] or ("TOTAL LINE" if i == total_at else "MERCHANT" if i == merchant_at else
                                         "DATE LINE" if i == date_at else "ignored")}
                   for i, (l, lab, p) in enumerate(zip(lines, labels, probs))],
            total={"value": total, "source": total_src, "line_index": total_at, "model_sureness": round(total_p, 2)},
            merchant={"value": out.get("merchant"), "line_index": merchant_at},
            date={"value": out.get("date"), "from_line": date_at, "note": date_note, "approx_from_bikram_sambat": bool(out.get("date_approx"))},
            arithmetic={"result": check, **detail},
            confidence={"model_total_sureness": round(total_p, 2), "model_mean_item_sureness": round(mean_p, 2),
                        "model_only": round(conf_model, 3), "rule_applied": rule, "final": out["confidence"],
                        "review_threshold": REVIEW_THRESHOLD})
    return out

## 4. OCR
docTR reads the photo. If the text it returns is too thin or looks like noise, Tesseract (English + Nepali) reads it too and the better reading wins.
Both give words with positions; `_rows` groups them into lines, top to bottom.

In [ ]:
# export order=2 name=ocr

# Colab ships TensorFlow too; make docTR use PyTorch (what we install) instead of guessing.

from PIL import Image, ImageOps

LANGS = os.environ.get("OCR_LANGS", "eng+nep")
PSM = os.environ.get("OCR_PSM", "6")      # 6 = one block of text; measured better than 4 on real phone photos
_cmd = os.environ.get("TESSERACT_CMD")     # Windows: C:\Program Files\Tesseract-OCR\tesseract.exe


def _tess():
    import pytesseract

    if _cmd:
        pytesseract.pytesseract.tesseract_cmd = _cmd
    return pytesseract


def available_langs() -> str:
    """The requested languages that are actually installed (so a missing Nepali pack is not fatal)."""
    have = set(_tess().get_languages(config=""))
    use = [l for l in LANGS.split("+") if l in have]
    return "+".join(use) or "eng"


def _flatten_background(img: Image.Image) -> Image.Image:
    """Divide by a blurred copy of the page: removes shadows and creases from phone photos.
    Needs OpenCV; without it the image is returned unchanged (OCR still works, a bit worse)."""
    try:
        import cv2
        import numpy as np
    except ImportError:
        return img
    a = np.array(img)
    bg = cv2.medianBlur(cv2.dilate(a, np.ones((15, 15), np.uint8)), 31)
    n = cv2.divide(a, bg, scale=255)
    return Image.fromarray(cv2.normalize(n, None, 0, 255, cv2.NORM_MINMAX))


def prepare(image: Image.Image) -> Image.Image:
    img = ImageOps.exif_transpose(image).convert("L")
    img = _flatten_background(img) if os.environ.get("OCR_FLATTEN", "1") == "1" else ImageOps.autocontrast(img, cutoff=1)
    w = img.width
    if w < 1200:
        img = img.resize((1200, round(img.height * 1200 / w)), Image.LANCZOS)
    elif w > 2400:
        img = img.resize((2400, round(img.height * 2400 / w)), Image.LANCZOS)
    return img


def _tesseract_lines(image: Image.Image, min_conf: int = 30):
    """Tesseract: photo in, list of text lines (top to bottom) out."""
    t = _tess()
    data = t.image_to_data(prepare(image), lang=available_langs(), config=f"--oem 1 --psm {PSM}",
                           output_type=t.Output.DICT)
    words = []
    for i, text in enumerate(data["text"]):
        text = text.strip()
        try:
            conf = float(data["conf"][i])
        except ValueError:
            conf = -1
        if text and conf >= min_conf:
            words.append((data["top"][i] + data["height"][i] / 2, data["left"][i], data["height"][i], text))
    if not words:
        return []
    return _rows(words)


def _rows(words):
    """words: (y_centre, x_left, height, text) -> text lines, top to bottom, words left to right."""
    words = sorted(words)
    tol = 0.6 * statistics.median(w[2] for w in words)
    rows, cur = [], [words[0]]
    for w in words[1:]:
        if abs(w[0] - statistics.mean(x[0] for x in cur)) <= tol:
            cur.append(w)
        else:
            rows.append(cur)
            cur = [w]
    rows.append(cur)
    return [" ".join(x[3] for x in sorted(r, key=lambda x: x[1])) for r in rows]


# --- docTR (open-source, self-hosted: db_resnet50 text detector + crnn_vgg16_bn recogniser) -------------
ENGINE = os.environ.get("OCR_ENGINE", "auto")      # auto = docTR if installed (plus Tesseract rescue), "doctr", "tesseract"
_doctr_model = None
last_engine = None          # which engine produced the lines of the last ocr_lines call


def doctr_available() -> bool:
    try:
        import doctr  # noqa: F401
        return True
    except ImportError:
        return False


def _doctr():
    global _doctr_model
    if _doctr_model is None:
        from doctr.models import ocr_predictor
        _doctr_model = ocr_predictor(det_arch="db_resnet50", reco_arch="crnn_vgg16_bn", pretrained=True)
    return _doctr_model


def _doctr_lines(image: Image.Image, max_side: int = 2200, min_conf: float = 0.2):
    import numpy as np
    img = ImageOps.exif_transpose(image).convert("RGB")
    s = max_side / max(img.size)
    if s < 1:
        img = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)
    page = _doctr()([np.array(img)]).pages[0]
    H, W = img.height, img.width
    words = []
    for block in page.blocks:
        for line in block.lines:
            for w in line.words:
                (x0, y0), (x1, y1) = w.geometry
                if w.value.strip() and w.confidence >= min_conf:
                    words.append(((y0 + y1) / 2 * H, x0 * W, max((y1 - y0) * H, 1.0), w.value.strip()))
    return _rows(words) if words else []


def _price_lines(lines) -> int:
    return sum(line_amount(l) is not None and any(c.isalpha() for c in l) for l in lines)


def ocr_lines(image: Image.Image, min_conf: int = 30):
    """Photo in, list of text lines (top to bottom) out. THIS IS THE OCR AI STEP (docTR, with Tesseract as the
    Nepali / rescue path): `ocr_lines` in this file.
    auto: docTR reads the photo; if it finds almost no "name  amount" lines (handwriting, mostly Devanagari bills)
    Tesseract eng+nep reads it too and the reading with more such lines wins."""
    global last_engine
    if ENGINE == "tesseract" or (ENGINE == "auto" and not doctr_available()):
        last_engine = "Tesseract"
        return _tesseract_lines(image, min_conf)
    try:
        lines = _doctr_lines(image)
    except Exception as e:
        if ENGINE == "doctr":
            raise
        print(f"[ocr] docTR failed ({type(e).__name__}: {str(e)[:120]}); using Tesseract instead")   # a broken docTR must not take the reader down
        last_engine = f"Tesseract (docTR failed: {type(e).__name__})"
        return _tesseract_lines(image, min_conf)
    last_engine = "docTR"
    if ENGINE == "auto" and (_price_lines(lines) < 3 or not looks_like_receipt(lines)):
        try:
            alt = _tesseract_lines(image, min_conf)
            better = (looks_like_receipt(alt) and not looks_like_receipt(lines)) or _price_lines(alt) > _price_lines(lines)
            if better:
                last_engine = "Tesseract (rescue: docTR text was too thin or looked like noise)"
                return alt
        except Exception:
            pass
    return lines

## 5. The two trained models
* **Line model**: for every OCR line, how likely it is the merchant, the date, an item, a subtotal, tax, the total or something else. Character n-grams plus a few layout numbers into a logistic regression.
* **Category model**: item name to one of eight categories. Character n-grams, so a badly spelled name still lands near the right category. If it is not clearly sure it answers **Other** (settings: `CATEGORY_MIN_PROB`, `CATEGORY_MIN_MARGIN`).

Both are tiny (about 1 MB) and train in seconds on a CPU.

In [ ]:
# export order=3 name=models
import joblib
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



_DEV = str.maketrans("०१२३४५६७८९", "0123456789")
_AMOUNT_END = re.compile(r"\d[\d,]*(?:\.\d{1,2})?\s*(?:/-|-)?\s*$")
_DATE = re.compile(r"\d{1,4}\s*[-/.]\s*\d{1,2}\s*[-/.]\s*\d{1,4}|\d{1,2}\s*[a-z]{3}[a-z]*\s*,?\s*\d{4}", re.I)
_CURRENCY = re.compile(r"(?i)\b(?:rs|npr)\b\.?|रु\.?")


def shape(text: str) -> str:
    """Lower-case, every run of digits becomes one '0', so '1,118.70' and '45.00' look alike."""
    t = text.translate(_DEV).lower()
    return re.sub(r"\d+", "0", t)


def numeric_features(lines):
    """One row per line: position in the receipt and what the line is made of."""
    n = len(lines)
    rows = []
    for i, raw in enumerate(lines):
        t = raw.translate(_DEV)
        L = max(len(t), 1)
        digits = sum(c.isdigit() for c in t)
        letters = sum(unicodedata.category(c)[0] in "LM" for c in t)
        upper = sum(c.isupper() for c in t)
        rows.append([
            1.0 if _AMOUNT_END.search(t) else 0.0,
            i / max(n - 1, 1),                       # 0 at the top, 1 at the bottom
            min(i, 8) / 8,                           # lines from the top / bottom, capped, so short and long receipts agree
            min(n - 1 - i, 8) / 8,
            digits / L,
            letters / L,
            min(L, 60) / 60,
            upper / max(letters, 1),
            min(len(re.findall(r"\d[\d,.]*", t)), 6) / 6,
            1.0 if _DATE.search(t) else 0.0,
            1.0 if _CURRENCY.search(t) else 0.0,
            1.0 if _NOT_ITEM.search(t) else 0.0,     # contains a total / tax / cash / date word (the same list the rules use)
        ])
    return np.array(rows, dtype=float)



LABELS = ["merchant", "date", "item", "subtotal", "tax", "total", "other"]


class LineModel:
    def __init__(self):
        self.chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True, max_features=40000)
        self.words = TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        self.clf = LogisticRegression(C=8.0, max_iter=3000, class_weight="balanced")

    def _x(self, lines, fit=False):
        shaped = [shape(l) for l in lines]
        f = (self.chars.fit_transform if fit else self.chars.transform)
        g = (self.words.fit_transform if fit else self.words.transform)
        return hstack([f(shaped), g(shaped), csr_matrix(numeric_features(lines))], format="csr")

    def fit(self, receipts):
        """receipts: list of [(line_text, label), ...], one list per receipt, top to bottom."""
        lines, y = [], []
        # numeric features depend on position inside the receipt, so build them per receipt
        feats = []
        for r in receipts:
            texts = [t for t, _ in r]
            feats.append(numeric_features(texts))
            lines += texts
            y += [lab for _, lab in r]
        shaped = [shape(l) for l in lines]
        X = hstack([self.chars.fit_transform(shaped), self.words.fit_transform(shaped), csr_matrix(np.vstack(feats))], format="csr")
        self.clf.fit(X, y)
        return self

    def predict_proba(self, lines):
        P = self.clf.predict_proba(self._x(lines))
        classes = list(self.clf.classes_)
        return [{lab: float(P[i, classes.index(lab)]) if lab in classes else 0.0 for lab in LABELS} for i in range(len(lines))]

    def predict(self, lines):
        return [max(p, key=p.get) for p in self.predict_proba(lines)]

    def save(self, path):
        joblib.dump(self, path, compress=3)

    @staticmethod
    def load(path):
        return joblib.load(path)



CATEGORIES = ["Groceries", "Dining", "Transport", "Shopping", "Health", "Bills", "Entertainment", "Other"]


class CategoryModel:
    def __init__(self):
        self.chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
        self.words = TfidfVectorizer(analyzer="word", token_pattern=r"\S+", sublinear_tf=True)
        self.clf = LogisticRegression(C=15.0, max_iter=3000, class_weight="balanced")

    def _x(self, names, fit=False):
        t = [shape(n) for n in names]
        if fit:
            return hstack([self.chars.fit_transform(t), self.words.fit_transform(t)], format="csr")
        return hstack([self.chars.transform(t), self.words.transform(t)], format="csr")

    def fit(self, rows):
        """rows: [(item_name, category), ...]"""
        self.clf.fit(self._x([n for n, _ in rows], fit=True), [c for _, c in rows])
        return self

    def predict(self, names):
        """-> [(category, probability)]. When the model is not clearly sure (best probability under CATEGORY_MIN_PROB, or
        less than CATEGORY_MIN_MARGIN ahead of the runner-up) the item is "Other": a wrong confident label is worse than a
        blank one the user fills in."""
        if not names:
            return []
        P = self.clf.predict_proba(self._x(names))
        out = []
        for row in P:
            order = row.argsort()[::-1]
            k, k2 = int(order[0]), int(order[1]) if len(order) > 1 else None
            cat, p = str(self.clf.classes_[k]), float(row[k])
            p2 = float(row[k2]) if k2 is not None else 0.0
            unsure = p < CATEGORY_MIN_PROB or (p - p2) < CATEGORY_MIN_MARGIN
            out.append(("Other" if unsure else cat, p))
        return out

    def predict_top(self, names, k=3):
        """-> per name, the k most likely [(category, probability)], for explaining a result."""
        if not names:
            return []
        P = self.clf.predict_proba(self._x(names))
        return [[(str(self.clf.classes_[j]), round(float(row[j]), 2)) for j in row.argsort()[::-1][:k]] for row in P]

    def save(self, path):
        joblib.dump(self, path, compress=3)

    @staticmethod
    def load(path):
        return joblib.load(path)

### The photo-to-receipt function
`read_photo` is the whole product in one call: OCR, then line model, then category model. `KharchaReader` is the small wrapper the backend will use.

In [ ]:
# export order=4 name=pipeline
from PIL import Image


class ModelsMissing(RuntimeError):
    """The trained models are not there. The reader refuses to run without them (the "delete the AI" proof)."""


class Models:
    def __init__(self, line, category):
        self.line = line
        self.category = category

    def save(self, folder):
        os.makedirs(folder, exist_ok=True)
        self.line.save(os.path.join(folder, "line_model.joblib"))
        self.category.save(os.path.join(folder, "category_model.joblib"))


def load_models(folder):
    """Both trained models from a folder, or None if the files are missing or were saved by another scikit-learn version."""
    lp, cp = os.path.join(folder, "line_model.joblib"), os.path.join(folder, "category_model.joblib")
    if not (os.path.exists(lp) and os.path.exists(cp)):
        return None
    try:
        m = Models(LineModel.load(lp), CategoryModel.load(cp))
        m.line.predict_proba(["Total 100.00"])        # use each once now: a model from another scikit-learn version
        m.category.predict(["rice"])                  # loads fine and then fails on its first real use
        return m
    except Exception as e:
        print(f"Could not use the trained models in {folder} ({type(e).__name__}: {e}). Train them again.")
        return None


def read_lines(lines, models, trace=None):
    """OCR lines -> receipt dict. models=None runs the plain rules only. trace: pass a dict to get the full reasoning back."""
    lines = [l.strip() for l in lines if l and l.strip()]
    if not looks_like_receipt(lines):
        if trace is not None:
            trace.update(unreadable=True, lines=[{"index": i, "text": l, "model_label": "-", "model_sureness": 0.0, "top3": {}, "role": "ignored"}
                                                 for i, l in enumerate(lines)], total={}, merchant={}, date={}, arithmetic={}, categories=[],
                         confidence={"final": 0.0, "review_threshold": REVIEW_THRESHOLD, "rule_applied": "text is noise, nothing was trusted"})
        return {"total": None, "items": [], "confidence": 0.0, "needs_review": True, "readable": False,
                "reconciliation": "The photo could not be read: the text looks like noise (handwriting, or a script the OCR cannot read). Enter it by hand."}
    out = parse_receipt(lines, models.line if models else None, trace)
    if models and out["items"]:
        names = [i["name"] for i in out["items"]]
        for item, (cat, _) in zip(out["items"], models.category.predict(names)):
            item["category"] = cat
        if trace is not None:
            trace["categories"] = [{"item": n, "top3": dict(t)} for n, t in zip(names, models.category.predict_top(names))]
    return out


def read_photo(image, models, trace=None):
    """Photo (PIL image or file path) -> {merchant, date, total, items[{name, qty, price, category}], confidence, needs_review}.
    The whole product in one call: OCR, then the trained line model, then the category model.
    models=None runs the rules-only baseline; a Models object is the normal case."""
    if isinstance(image, (str, os.PathLike)):
        image = Image.open(image)
    lines = ocr_lines(image)
    if trace is not None:
        trace["ocr_engine"] = last_engine
        trace["ocr_lines"] = lines
    return read_lines(lines, models, trace)


class KharchaReader:
    """What the backend uses:  reader = KharchaReader("folder with the two .joblib files");  reader.read("photo.jpg")"""

    def __init__(self, folder="."):
        self.models = load_models(folder)
        if self.models is None:
            raise ModelsMissing(f"No usable trained models in {folder!r}.")

    def read(self, image, trace=None):
        return read_photo(image, self.models, trace)

## 6. Training data and how to train
**Fake receipts** (`make_dataset`): receipts generated with known answers, English and Nepali, with OCR-style typos. Good for a first model, but easier than real life.
**Your real bills**: the 25 photos in `Bills.zip`, read by the OCR and labelled automatically from the answer key (`bills_gold.json`, read by eye from the photos, so it can contain mistakes).

In [ ]:
# Training data, part 1: the item names per category (starter vocabulary)
CATEGORY_NAMES = {
    "Groceries": [
        "basmati rice 5kg", "rice 25kg", "chamal", "चामल", "jeera masino rice", "lentils 2kg", "dal", "दाल", "masoor dal",
        "cooking oil 5l", "mustard oil", "tel", "तेल", "sunflower oil", "toned milk 1l", "milk packet", "dudh", "दूध",
        "eggs tray", "egg 12 pcs", "anda", "अण्डा", "vegetables", "tarkari", "तरकारी", "potato 2kg", "onion 1kg", "tomato",
        "garlic ginger", "fruits", "apple 1kg", "banana dozen", "orange", "falful", "फलफूल", "atta 10kg", "आटा", "maida flour",
        "sugar 2kg", "chini", "चिनी", "salt packet", "nun", "नुन", "tea 500g", "tea leaves", "butter 500g", "ghee 1l",
        "ghiu", "घिउ", "dairy", "curd 1kg", "dahi", "दही", "paneer", "bread loaf", "brown bread", "pau roti", "noodles pack",
        "wai wai", "वाइवाइ", "biscuits", "snacks", "chips", "chicken 1kg", "mutton 1kg", "masu", "मासु", "fish", "spices masala",
        "turmeric powder", "chana", "rajma beans", "peanuts", "dry fruits", "honey", "jam", "pickle achar", "अचार",
    ],
    "Dining": [
        "momo", "mo:mo", "मोमो", "chicken momo", "buff momo", "veg momo", "jhol momo", "thali set", "थाली", "dal bhat set", "khana set",
        "खाना", "nasi goreng", "nasi campur", "mie goreng", "burger", "cheese burger", "pizza", "margherita pizza", "pasta",
        "chowmein", "चाउमिन", "chow mein", "fried rice", "chilli chicken", "sekuwa", "सेकुवा", "choila", "chatamari", "चटामरी",
        "sizzler", "steak", "wagyu ribeye", "sandwich", "club sandwich", "french fries", "samosa", "चिया", "chiya", "milk tea", "masala tea",
        "coffee", "कफी", "cappuccino", "latte", "espresso", "americano", "cold coffee", "ice tea", "lassi", "लस्सी", "mango lassi",
        "juice", "orange juice", "soft drink", "coke", "beer", "chhyang", "whisky", "cake slice", "chocolate cake", "brownies",
        "ice cream", "dessert", "service charge", "set breakfast", "puri tarkari", "jeri swari", "lunch", "dinner", "drinks",
    ],
    "Transport": [
        "petrol 30l", "petrol", "पेट्रोल", "diesel 20l", "डिजेल", "fuel", "taxi fare", "taxi", "pathao ride", "pathao", "indrive",
        "uber trip", "bus ticket", "micro fare", "tempo fare", "parking", "parking fee", "toll", "bike service", "tyre repair",
        "puncture", "engine oil", "bike wash", "airport transfer", "flight ticket", "bus pass", "ride fare", "vehicle tax",
        "ev charging", "cng", "स्कुटर सर्भिस",
    ],
    "Shopping": [
        "detergent", "washing powder", "soap", "साबुन", "dish soap", "shirt", "tshirt", "jeans", "कपडा", "lugha", "लुगा",
        "shoes", "जुत्ता", "sandals", "earphones", "earphone", "cable", "usb cable", "charger", "phone cover", "power bank",
        "bag", "backpack", "wallet", "watch", "cushion", "bedsheet", "towel", "utensils", "pressure cooker", "bulb",
        "extension cord", "notebook", "pen set", "gift item", "toy", "perfume", "jacket", "cap", "umbrella",
    ],
    "Health": [
        "vitamins", "vitamin c", "multivitamin", "paracetamol", "tablet", "capsule", "syrup", "cough syrup", "medicine", "औषधि",
        "pharmacy", "फार्मेसी", "antibiotic", "bandage", "face wash", "shampoo", "toothpaste", "toothbrush", "sanitizer",
        "mask", "thermometer", "doctor fee", "consultation", "clinic", "lab test", "blood test", "x-ray", "dental checkup",
        "eye drops", "ointment", "protein powder", "first aid", "pad", "diapers", "baby lotion", "ors",
    ],
    "Bills": [
        "electricity bill", "nea electricity", "बिजुली", "bijuli bill", "water bill", "khanepani", "खानेपानी", "internet bill",
        "wifi", "broadband", "worldlink", "subisu", "mobile top-up", "mobile topup", "recharge", "ntc recharge", "ncell recharge",
        "data pack", "phone bill", "cable tv", "dish home", "rent", "room rent", "भाडा", "house rent", "gas cylinder", "lpg",
        "insurance premium", "loan emi", "school fee", "tuition fee", "garbage fee", "tax payment", "subscription", "app subscription",
    ],
    "Entertainment": [
        "movie tickets", "movie ticket", "cinema", "qfx", "film ticket", "popcorn", "popcorn and drinks", "netflix", "spotify",
        "youtube premium", "game top-up", "pubg uc", "free fire diamonds", "concert ticket", "event pass", "bowling", "arcade",
        "karaoke", "museum entry", "zoo ticket", "amusement park", "gaming cafe", "book", "magazine", "ps5 game", "playstation",
        "cricket match ticket", "dj night", "club entry", "paragliding", "rafting", "trek permit",
    ],
    "Other": [
        "miscellaneous", "misc", "other", "sundry", "adjustment", "donation", "gift", "courier", "postage", "photocopy", "printing",
        "laundry", "repair", "service", "stamp", "xerox", "lamination", "locker fee", "handling fee",
        "chair", "wooden chair", "table", "sofa", "furniture", "almirah", "cupboard", "wooden bed", "desk", "stool", "shelf",
    ],
}


def seed_rows():
    return [(n, c) for c, names in CATEGORY_NAMES.items() for n in names]


# Training data, part 2: fake receipts with known answers

MERCHANTS_EN = ["Corner Mart", "Fresh Basket", "Hill View Cafe", "Metro Fuel", "City Pharmacy", "Sher Restaurant",
                "Big Mart", "Daily Needs Store", "Himal Bakery", "Everest Supermarket", "Green Valley Cafe", "Lake Side Kitchen"]
MERCHANTS_NP = ["नेपाल सुपरमार्केट", "हिमालय क्याफे", "काठमाडौं मार्ट", "नमस्ते रेस्टुरेन्ट", "पोखरा बेकरी", "सगरमाथा स्टोर"]
HELD_OUT_MERCHANTS = ["Tea Garden Mart", "Pashupati Stores", "गोरखा मार्ट", "Sunrise Pharmacy", "Thamel Diner"]
ADDRESSES = ["Kathmandu, Nepal", "Thamel, Kathmandu", "Lazimpat, Kathmandu", "Pokhara-8, Lakeside", "Patan, Lalitpur",
             "काठमाडौं, नेपाल", "ललितपुर", "Tel: 01-4412345", "Ph: 9841234567", "Bhaktapur"]
TOTAL_WORDS = ["Total", "TOTAL", "Grand Total", "Net Amount", "Amount Due", "Total Amount", "Net Payable", "जम्मा", "कुल जम्मा", "कुल रकम", "जम्मा रकम"]
SUBTOTAL_WORDS = ["Subtotal", "Sub Total", "SUB TOTAL", "Sub-total", "उप जम्मा", "उपजम्मा"]
TAX_WORDS = ["VAT 13%", "VAT", "Service Charge 10%", "Service Charge", "Discount", "Tax", "भ्याट १३%", "भ्याट", "सेवा शुल्क", "छुट"]
PAY_WORDS = ["Cash", "Change", "Paid", "Card", "Balance", "Tendered", "नगद", "फिर्ता", "भुक्तानी"]
OTHER_LINES = ["Thank you, visit again", "Thank you!", "धन्यवाद", "फेरि आउनुहोला", "Cashier: Ram", "Table 4", "Counter 2",
               "Item  Qty  Rate  Amount", "विवरण  दर  रकम", "Powered by Kharcha POS", "No refund without bill", "Please keep this bill"]
HEADER_PAN = ["PAN No: {n}", "VAT No. {n}", "प्यान नं: {n}", "PAN: {n}"]
HEADER_INV = ["Invoice No: {n}", "Bill No. {n}", "Receipt #{n}", "बिल नं: {n}"]
DATE_FORMS = ["Date: {d:02d}/{m:02d}/{y}", "{y}-{m:02d}-{d:02d} {h:02d}:{mi:02d}", "{d} {mon} {y}", "Date {d:02d}-{m:02d}-{y}", "मिति: {y}/{m:02d}/{d:02d}", "{d:02d}.{m:02d}.{y}"]
MONTHS = "Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split()
NP_DIGITS = str.maketrans("0123456789", "०१२३४५६७८९")


def item_pool(held_out: bool):
    """Item names per category. Even positions train, odd positions test, so tests use unseen names."""
    pool = []
    for cat, names in CATEGORY_NAMES.items():
        for i, n in enumerate(names):
            if (i % 2 == 1) == held_out:
                pool.append((n, cat))
    return pool


def _money(rng, x):
    style = rng.random()
    if style < 0.4:
        return f"{x:,.2f}"
    if style < 0.7:
        return f"{x:.2f}"
    return f"{x:,.0f}" if float(x).is_integer() else f"{x:.2f}"


def _noise(rng, text, p):
    if rng.random() > p:
        return text
    chars = list(text)
    for _ in range(rng.choice([1, 1, 2, 3])):
        if not chars:
            break
        i = rng.randrange(len(chars))
        op = rng.random()
        # Real OCR confuses look-alike characters (o/0, l/1) far more often than it drops digits,
        # so deleting or inserting inside numbers is mostly skipped.
        if chars[i].isdigit() and op >= 0.4 and rng.random() < 0.85:
            continue
        if op < 0.4:
            chars[i] = {"o": "0", "0": "o", "l": "1", "1": "l", "s": "5", "i": "l", "a": "e", "e": "c"}.get(chars[i].lower(), chars[i])
        elif op < 0.7:
            del chars[i]
        else:
            chars.insert(i, rng.choice(" .,'"))
    return "".join(chars)


def make_receipt(rng: random.Random, held_out=False, noise=0.15):
    """-> (labelled lines [(text, label)], gold dict)."""
    np_style = rng.random() < 0.35
    merchants = HELD_OUT_MERCHANTS if held_out else (MERCHANTS_NP if np_style and rng.random() < 0.6 else MERCHANTS_EN + MERCHANTS_NP)
    merchant = rng.choice(merchants)
    pool = item_pool(held_out)
    L = []

    def add(text, label):
        if np_style and rng.random() < 0.5:
            text = text.translate(NP_DIGITS)
        L.append((_noise(rng, text, noise), label))

    add(merchant if rng.random() < 0.5 else merchant.upper(), "merchant")
    if rng.random() < 0.8:
        add(rng.choice(ADDRESSES), "other")
    if rng.random() < 0.6:
        add(rng.choice(HEADER_PAN).format(n=rng.randrange(10**8, 10**9)), "other")
    y, m, d = rng.choice([2025, 2026]), rng.randint(1, 12), rng.randint(1, 28)
    date = f"{y:04d}-{m:02d}-{d:02d}"
    if rng.random() < 0.9:
        add(rng.choice(DATE_FORMS).format(y=y, m=m, d=d, h=rng.randint(8, 21), mi=rng.randint(0, 59), mon=MONTHS[m - 1]), "date")
    if rng.random() < 0.5:
        add(rng.choice(HEADER_INV).format(n=rng.randint(100, 99999)), "other")
    if rng.random() < 0.5:
        add(rng.choice(["Item  Qty  Rate  Amount", "विवरण  दर  रकम", "Description    Amount", "ITEM        PRICE"]), "other")

    items, subtotal = [], 0.0
    for name, cat in rng.sample(pool, rng.randint(1, 8)):
        qty = rng.choice([1, 1, 1, 2, 3])
        unit = float(rng.choice([20, 35, 45, 60, 85, 90, 120, 150, 210, 250, 380, 450, 640, 850, 1250, 2400])) * rng.choice([1, 1, 1.5])
        line_total = round(unit * qty, 2)
        shown = name.title() if rng.random() < 0.3 else name
        fmt = rng.randrange(6)
        if qty == 1 or fmt == 0:
            if fmt == 0 and qty > 1:
                text = f"{shown}   {_money(rng, line_total)}"
            else:
                text = {1: f"{shown}   {_money(rng, line_total)}", 2: f"{shown}  Rs. {_money(rng, line_total)}",
                        3: f"{shown} ........ {_money(rng, line_total)}", 4: f"{shown:<24}{_money(rng, line_total)}",
                        5: f"{shown}  रु {_money(rng, line_total)}"}.get(fmt, f"{shown}   {_money(rng, line_total)}")
        elif fmt == 1:
            text = f"{qty} x {shown}   {_money(rng, unit)}   {_money(rng, line_total)}"
        elif fmt == 2:
            text = f"{shown}   {qty} x {_money(rng, unit)}   {_money(rng, line_total)}"
        elif fmt == 3:
            text = f"{qty} {shown}   {_money(rng, line_total)}"
        else:
            text = f"{shown} {qty}x{_money(rng, unit)}  {_money(rng, line_total)}"
        add(text, "item")
        items.append({"name": shown, "qty": qty, "price": line_total, "category": cat})
        subtotal += line_total

    total = subtotal
    if rng.random() < 0.6:
        add(f"{rng.choice(SUBTOTAL_WORDS)}   {_money(rng, subtotal)}", "subtotal")
    for _ in range(rng.choice([0, 0, 1, 1, 2])):
        word = rng.choice(TAX_WORDS)
        amt = round(subtotal * rng.choice([0.05, 0.10, 0.13]), 2)
        neg = word in ("Discount", "छुट")
        total += -amt if neg else amt
        add(f"{word}   {'-' if neg else ''}{_money(rng, amt)}", "tax")
    total = round(total, 2)
    add(f"{rng.choice(TOTAL_WORDS)}   {_money(rng, total)}", "total")
    for _ in range(rng.choice([0, 1, 2])):
        add(f"{rng.choice(PAY_WORDS)}   {_money(rng, total + rng.choice([0, 50, 100, 500]))}", "other")
    for _ in range(rng.choice([0, 1, 2])):
        add(rng.choice(OTHER_LINES), "other")
    gold = {"merchant": merchant, "date": date, "total": total, "items": items}
    return L, gold


def make_dataset(n, seed=0, held_out=False, noise=0.15):
    rng = random.Random(seed)
    return [make_receipt(rng, held_out, noise) for _ in range(n)]


def _font(size):
    """A monospace font that exists on this machine (Colab and Linux paths, matplotlib's bundled DejaVu, Windows/macOS, last resort PIL's own)."""
    import glob
    import os
    from PIL import ImageFont
    cands = ["/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", "/usr/share/fonts/dejavu/DejaVuSansMono.ttf",
             "/usr/share/fonts/TTF/DejaVuSansMono.ttf", "/Library/Fonts/Courier New.ttf", "C:/Windows/Fonts/consola.ttf"]
    try:
        import matplotlib
        cands.append(os.path.join(matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSansMono.ttf"))
    except Exception:
        pass
    cands += glob.glob("/usr/share/fonts/**/*Mono*.ttf", recursive=True) + glob.glob("/usr/share/fonts/**/DejaVuSans.ttf", recursive=True)
    for p in cands:
        if os.path.exists(p):
            try:
                return ImageFont.truetype(p, size)
            except OSError:
                continue
    return ImageFont.load_default(size)          # Pillow >= 10.1

In [ ]:
# The answer key for the 25 bills (read by eye from the photos, not from the OCR). Categories marked ? are guesses.
GOLD_JSON = r'''{
  "_about": "Answers for the 25 photos in Bills.zip, read by eye from the photos (not from the OCR). Categories are one person's judgement: check the ones marked ? before trusting them. total=null means the photo is cut off before the total. Dates: date_ad is the Gregorian date printed on the bill, 'MM/DD/YYYY' on these POS printers; date_bs is the Bikram Sambat date as printed (DD/MM/YYYY).",
  "same_photo_as": {"24.jpg": "1.jpg", "23.jpg": "2.jpg", "22.jpg": "3.jpg", "17.jpg": "16.jpg"},
  "receipts": {
    "1.jpg":  {"merchant": "Binayak Furniture", "date_bs": "2082-04-18", "total": 4802.50, "handwritten": true,
               "items": [{"name": "Chair (handwritten)", "price": 4250, "category": "Shopping"}]},
    "2.jpg":  {"merchant": "Meridian International School", "date_ad": "2026-06-19", "total": 9200,
               "items": [{"name": "Academic Resource fee", "price": 9200, "category": "Other"}]},
    "3.jpg":  {"merchant": "Bhat-Bhateni Super Market (Maharajgunj)", "date_bs": "23/06/2083", "total": null, "items": null,
               "note": "cut off before the total; 28+ items, not written out here"},
    "4.jpg":  {"merchant": "Bhat-Bhateni Super Market (Tangal)", "date_bs": "12/06/2083", "total": 125, "subtotal": 145,
               "items": [{"name": "COSTA BRAVA", "price": 145, "category": "Groceries?"}]},
    "5.jpg":  {"merchant": "Bhat-Bhateni Super Market (Tangal)", "date_bs": "16/06/2083", "total": 90,
               "items": [{"name": "UC CHICKEN", "price": 90, "category": "Groceries"}]},
    "6.jpg":  {"merchant": "Pepe Pizza Restaurant and Bar Pvt. Ltd.", "date_ad": "2026-10-02", "total": 555, "subtotal": 491.15, "tax": 63.85,
               "items": [{"name": "BUFF STEAM MOMO", "price": 491.15, "category": "Dining"}]},
    "7.jpg":  {"merchant": "Big Mart", "date_ad": "2026-09-29", "total": 344,
               "items": [{"name": "CENTURY NACHOZ HOT&SPICY", "price": 50, "category": "Groceries"},
                         {"name": "BRITANNIA FRUIT CAKE", "price": 230, "category": "Groceries"},
                         {"name": "NESTLE MILKY BAR MOOSHA", "price": 64, "category": "Groceries"}]},
    "8.jpg":  {"merchant": "G.R.G Bakery Pvt. Ltd", "date_ad": "2026-10-12", "total": 340,
               "items": [{"name": "blueberry cheesecake pastry", "price": 190, "category": "Dining"},
                         {"name": "plain cheesecake pastry", "price": 150, "category": "Dining"}]},
    "9.jpg":  {"merchant": "Krispy Krunchy Fried Chicken Pvt. Ltd.", "date_ad": "2026-10-07", "total": 999, "subtotal": 884.07, "tax": 114.93,
               "items": [{"name": "WEDNESDAY BIG BUCKET", "price": 884.07, "category": "Dining"}]},
    "10.jpg": {"merchant": "Bhat-Bhateni Super Market (Maharajgunj)", "date_bs": "23/06/2083", "total": 1915,
               "items": [{"name": "REGAN STEEL", "price": 1600, "category": "Shopping"},
                         {"name": "GUANGBO ACR", "price": 315, "category": "Shopping?"}]},
    "11.jpg": {"merchant": "Bhat-Bhateni Super Market (Maharajgunj)", "date_bs": "23/06/2083", "total": 3033,
               "items": [{"name": "FASHION BAR", "price": 1435, "category": "Shopping?"},
                         {"name": "REVLON SUPE", "price": 1598, "category": "Shopping"}]},
    "12.jpg": {"merchant": "Bhat-Bhateni Super Market (Maharajgunj)", "date_bs": "23/06/2083", "total": 2635,
               "items": [{"name": "FASHION MEN", "price": 2635, "category": "Shopping"}]},
    "13.jpg": {"merchant": "Bhat-Bhateni Super Market (Tangal)", "date_bs": "16/06/2083", "total": 1404.60, "subtotal": 1419,
               "items": [{"name": "URBAN CHICK", "price": 245, "category": "Groceries"},
                         {"name": "URBAN BUFF", "price": 400, "category": "Groceries"},
                         {"name": "JABSONS PEA", "price": 58, "category": "Groceries"},
                         {"name": "CADBURY DAI", "price": 100, "category": "Groceries"},
                         {"name": "BERRAK SPAG", "price": 180, "category": "Groceries"},
                         {"name": "BAHAR PASTA", "price": 190, "category": "Groceries"},
                         {"name": "HIMALAYA PU", "price": 144, "category": "Health?"},
                         {"name": "HARBAL GAJA", "price": 102, "category": "Other?"}]},
    "14.jpg": {"merchant": "Pepe Pizza Restaurant and Bar Pvt. Ltd.", "date_ad": "2026-10-02", "total": 350, "subtotal": 309.73, "tax": 40.27,
               "items": [{"name": "MARGHERITA PIZZA", "price": 309.73, "category": "Dining"}]},
    "15.jpg": {"merchant": "Bhat-Bhateni Super Market (Tangal)", "date_bs": "05/06/2083", "total": 2085,
               "items": [{"name": "DHARA HEALT", "price": 341, "category": "Groceries"},
                         {"name": "DHARA KACHI", "price": 401, "category": "Groceries"},
                         {"name": "TIDE + EXTR", "price": 310, "category": "Shopping"},
                         {"name": "NIVEA SOFT", "price": 361, "category": "Health"},
                         {"name": "KANISHA COT", "price": 55, "category": "Shopping?"},
                         {"name": "STC SALT 1K", "price": 25, "category": "Groceries"},
                         {"name": "BBSM BESAN", "price": 125, "category": "Groceries"},
                         {"name": "ORGANIC MIL", "price": 146, "category": "Groceries?"},
                         {"name": "MANGALDEEP", "price": 100, "category": "Shopping?"},
                         {"name": "VIM BAR 100", "price": 40, "category": "Shopping"},
                         {"name": "TIDE WHITE", "price": 96, "category": "Shopping"},
                         {"name": "SHAGUN YELL", "price": 85, "category": "Groceries?"}]},
    "16.jpg": {"merchant": "Bhat-Bhateni Super Market (Boudha)", "date_bs": "04/06/2083", "total": 695,
               "items": [{"name": "TY CLOTHES", "price": 305, "category": "Shopping"},
                         {"name": "FABER CASTE", "price": 120, "category": "Shopping"},
                         {"name": "NOTE BOOK X", "price": 135, "category": "Shopping"},
                         {"name": "GUANGBO SUP", "price": 40, "category": "Shopping?"},
                         {"name": "KRITI ARTIS", "price": 95, "category": "Shopping?"}]},
    "18.jpg": {"merchant": "Bhat-Bhateni Super Market (Tangal)", "date_bs": "16/06/2083", "total": 3648,
               "items": [{"name": "LADY HALF T", "price": 1729, "category": "Shopping"},
                         {"name": "LOTUS BABY", "price": 1919, "category": "Shopping?"}]},
    "19.jpg": {"merchant": "Bhat-Bhateni Super Market (Tangal)", "date_bs": "16/06/2083", "total": 1784,
               "items": [{"name": "LUX COZY ME", "price": 229, "category": "Shopping"},
                         {"name": "ZOOM PLUS M", "price": 1555, "category": "Shopping?"}]},
    "20.jpg": {"merchant": "Big Mart", "date_ad": "2026-09-13", "total": 615,
               "items": [{"name": "KHAJURICO PUFF REGULAR GR", "price": 70, "category": "Groceries"},
                         {"name": "NEWARI SHAHI PULAO BASMATI", "price": 225, "category": "Groceries"},
                         {"name": "NESTLE KITKAT", "price": 80, "category": "Groceries"},
                         {"name": "BRITANNIA 50-50 TIME PASS", "price": 120, "category": "Groceries"},
                         {"name": "BRITANNIA 50-50 SWEET & SALTY", "price": 120, "category": "Groceries"}]},
    "21.jpg": {"merchant": "न्यू मिश्र राईस मिल", "total": null, "handwritten": true,
               "items": [{"name": "rice (handwritten)", "price": null, "category": "Groceries"}],
               "note": "handwritten Nepali; amounts could not be read reliably"},
    "25.jpg": {"merchant": "Sun International Incorporated Pvt. Ltd.", "date_ad": "2026-07-09", "total": 1205, "subtotal": 1066.37, "tax": 138.63,
               "items": [{"name": "REGULAR SCOOP", "price": 345.13, "category": "Dining"},
                         {"name": "REGULAR SCOOP", "price": 345.13, "category": "Dining"},
                         {"name": "REGULAR SCOOP", "price": 376.11, "category": "Dining"}]}
  }
}
'''

In [ ]:
# Helpers for the real bills: load the answer key, label OCR lines from it, split into folds, judge a read against the key
import difflib



def _bs_date(s):
    """'23/06/2083' or '2082-04-18' -> approximate Gregorian ISO date."""
    return find_date_bs(s or "")


def load_gold():
    """{photo: {merchant, date, total, items[{name, price, category}], group, handwritten, scorable}}"""
    raw = json.loads(GOLD_JSON)
    same = raw.get("same_photo_as", {})
    out = {}
    for name, r in raw["receipts"].items():
        date = r.get("date_ad") or _bs_date(r.get("date_bs"))
        items = None
        if r.get("items") is not None:
            items = [{"name": i["name"], "price": float(i["price"]), "category": i.get("category", "Other").rstrip("?")}
                     for i in r["items"] if i.get("price") is not None]
        out[name] = {"merchant": r.get("merchant"), "date": date, "date_is_bs": not r.get("date_ad") and bool(date),
                     "total": r.get("total"), "items": items, "group": name, "handwritten": bool(r.get("handwritten")),
                     "scorable": r.get("total") is not None and items is not None and not r.get("handwritten")}
    for dup, orig in same.items():                  # the duplicate photo shares everything with its original
        out[dup] = {**out[orig], "group": orig}
    return out


def _sim(a, b):
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()


def autolabel(lines, gold):
    """Label each OCR line from the answer key. Returns (labels, categories) lists, one entry per line."""
    n = len(lines)
    labels, cats = ["other"] * n, [""] * n
    want = list(gold.get("items") or [])
    short = (gold.get("merchant") or "").split("(")[0].strip()
    best = (0.0, None)
    for i, l in enumerate(lines[:8]):
        txt = re.sub(r"[^A-Za-z ]", "", l).strip()
        if len(txt) >= 3 and short:
            best = max(best, (_sim(txt, short), i), key=lambda x: x[0])
    if best[1] is not None and best[0] >= 0.5:
        labels[best[1]] = "merchant"
    for i, l in enumerate(lines):
        t = normalize(l)
        if labels[i] != "other":
            continue
        if find_date(t) or find_date_bs(t):
            labels[i] = "date"
        elif re.search(r"(?i)sub\s*-?\s*total|gross", t) and line_amount(l):
            labels[i] = "subtotal"
        elif re.search(r"(?i)\bvat\b|\btax\b", t) and not _TOTAL_WORDS.search(t) and line_amount(l):
            labels[i] = "tax"
    total = gold.get("total")
    if total is not None:
        for i in reversed(range(n)):
            if labels[i] != "other" or not (_TOTAL_WORDS.search(lines[i]) and not _NOT_TOTAL.search(lines[i])):
                continue
            amt = line_amount(lines[i])
            if amt is None and i + 1 < n:
                amt = line_amount(lines[i + 1])
            if amt is not None and abs(amt - total) < 0.01:
                labels[i] = "total"
                break
    for i, l in enumerate(lines):
        if labels[i] != "other":
            continue
        it = parse_item_line(l, strict=True)
        if not it:
            continue
        hit = next((g for g in want if abs(g["price"] - it["price"]) < 0.01), None)
        if hit:
            want.remove(hit)
            labels[i], cats[i] = "item", hit.get("category", "")
    return labels, cats


def build_receipts(ocr_by_photo, gold):
    """OCR lines per photo + answers -> {photo: {"texts", "lines", "gold", "rows"}} (the shape train.py/label_tools use)."""
    out = {}
    for name, lines in ocr_by_photo.items():
        g = gold.get(name)
        if not g or not lines:
            continue
        lab, cat = autolabel(lines, g)
        rows = [{"receipt": name, "line": i, "text": t, "label": l, "category": c, "corrected": ""}
                for i, (t, l, c) in enumerate(zip(lines, lab, cat))]
        out[name] = {"texts": list(lines), "lines": list(zip(lines, lab)), "gold": g, "rows": rows}
    return out


def trainable(rec):
    """Bills worth training on: a known total and item list, not handwritten (handwriting is read as noise)."""
    return sorted(n for n, r in rec.items() if r["gold"]["scorable"])


def folds(names, gold, k=4, seed=0):
    """Split photo names into k folds, duplicates of one bill always together."""
    groups = sorted({gold[n]["group"] for n in names})
    random.Random(seed).shuffle(groups)
    gfold = {g: i % k for i, g in enumerate(groups)}
    return [[n for n in names if gfold[gold[n]["group"]] == f] for f in range(k)]


def judge(pred, gold):
    """Per-bill correctness: total, date (3 days of slack when it came from a Bikram Sambat date), all item prices."""
    pt = pred.get("total")
    total_ok = pt is not None and abs(pt - gold["total"]) <= max(0.01, 0.005 * gold["total"])
    date_ok = None
    if gold.get("date"):
        if pred.get("date") and gold["date_is_bs"]:
            from datetime import date as D
            date_ok = abs((D.fromisoformat(pred["date"]) - D.fromisoformat(gold["date"])).days) <= 3
        else:
            date_ok = pred.get("date") == gold["date"]
    got = sorted(round(i["price"], 2) for i in pred.get("items", []))
    items_ok = got == sorted(round(i["price"], 2) for i in gold["items"])
    merch_ok = bool(pred.get("merchant")) and _sim(pred["merchant"], gold["merchant"] or "") >= 0.6
    return {"total": total_ok, "date": date_ok, "items": items_ok, "merchant": merch_ok}


def diagnose(pred, gold, j=None):
    """One plain sentence on what went wrong with a read (or 'all correct')."""
    j = j or judge(pred, gold)
    why = []
    pt = pred.get("total")
    if not j["total"]:
        why.append("no total found" if pt is None else f"total read as {pt:g}, should be {gold['total']:g}")
    got = [round(i["price"], 2) for i in pred.get("items", [])]
    want = [round(i["price"], 2) for i in gold["items"]]
    if not j["items"]:
        missing = [w for w in want if w not in got]
        extra = [g for g in got if g not in want]
        part = []
        if missing:
            part.append(f"missed prices {missing[:4]}{'...' if len(missing) > 4 else ''}")
        if extra:
            part.append(f"extra junk items {extra[:3]}")
        if part:
            why.append("items: " + ", ".join(part))
    if j["date"] is False:
        why.append(f"date {pred.get('date')} should be {gold['date']}")
    if not j["merchant"]:
        why.append(f"store read as {pred.get('merchant')!r}")
    return "; ".join(why) or "all correct"


def price_recall(lines, gold):
    """(found, wanted): how many of the answer's item prices and its total show up as numbers in the OCR text."""
    nums = set(x.replace(",", "") for x in re.findall(r"\d[\d,]*\.\d{1,2}|\d[\d,]*", normalize(" ".join(lines))))

    def has(v):
        s = f"{v:.2f}"
        return s in nums or (s.endswith(".00") and s[:-3] in nums) or (s.endswith("0") and s[:-1] in nums)
    want = [i["price"] for i in (gold.get("items") or [])] + ([gold["total"]] if gold.get("total") is not None else [])
    return sum(has(v) for v in want), len(want)


def missing_numbers(lines, gold):
    """The answer-key amounts (items and total) that the OCR text does not contain, for explaining an OCR miss."""
    nums = set(x.replace(",", "") for x in re.findall(r"\d[\d,]*\.\d{1,2}|\d[\d,]*", normalize(" ".join(lines))))

    def has(v):
        s = f"{v:.2f}"
        return s in nums or (s.endswith(".00") and s[:-3] in nums) or (s.endswith("0") and s[:-1] in nums)
    out = [("item", i["price"]) for i in (gold.get("items") or []) if not has(i["price"])]
    if gold.get("total") is not None and not has(gold["total"]):
        out.append(("total", gold["total"]))
    return out

In [ ]:
def train_models(real_receipts=(), real_items=(), synthetic=4000, seed=1, receipt_weight=20, item_weight=5, quiet=False):
    """Fit both models and return a Models object.
    real_receipts: [[(line_text, label), ...], ...] your labelled bills; each counts `receipt_weight` times because fake receipts outnumber them.
    real_items: [(item_name, category), ...] from your bills; each counts `item_weight` times."""
    say = (lambda *a: None) if quiet else print
    receipts = [r for r, _ in make_dataset(synthetic, seed=seed)] + list(real_receipts) * receipt_weight
    bad = {l for r in receipts for _, l in r} - set(LABELS)
    assert not bad, f"unknown labels: {bad}"
    lm = LineModel().fit(receipts)
    rows = seed_rows() + list(real_items) * item_weight
    cm = CategoryModel().fit(rows)
    say(f"line model: trained on {len(receipts)} receipts ({synthetic} fake + {len(real_receipts)} real x{receipt_weight})")
    say(f"category model: trained on {len(rows)} items ({len(seed_rows())} starter names + {len(real_items)} from real bills x{item_weight})")
    return Models(lm, cm)


def bill_items(rec, names):
    return [(i["name"], i["category"]) for n in names for i in (rec[n]["gold"]["items"] or [])]

t0 = time.time()
STARTER = train_models(quiet=False)
print(f"starter models (fake receipts only) trained in {time.time() - t0:.0f} s")

## 7. Test 1: fake receipts the model has never seen
`held_out=True` uses merchants and item names that were **not** in the training data. Fake receipts are easier than real ones, so treat this as a sanity check, not as the real accuracy.

In [ ]:
from collections import Counter, defaultdict

def test_on_fake(models, n=500, seed=99):
    held = make_dataset(n, seed=seed, held_out=True)
    hit, tot = Counter(), Counter()
    right = 0
    for lines, _ in held:
        texts, labs = [t for t, _ in lines], [l for _, l in lines]
        for lab, pred in zip(labs, models.line.predict(texts)):
            tot[lab] += 1
            hit[lab] += lab == pred
            right += lab == pred
    N = sum(tot.values())
    print(f"LINE MODEL on {n} unseen fake receipts ({N} lines): {100 * right / N:.1f}% of lines labelled correctly")
    print(f"  {'label':10}{'lines':>7}{'correct':>9}")
    for lab in LABELS:
        if tot[lab]:
            print(f"  {lab:10}{tot[lab]:>7}{100 * hit[lab] / tot[lab]:>8.0f}%")
    names, cats = zip(*item_pool(True))
    pred = models.category.predict(list(names))
    sure = [(p[0], c) for p, c in zip(pred, cats) if p[0] != "Other" or c == "Other"]
    answered = [(p, c) for (p, _), c in zip(pred, cats) if p != "Other"]
    ok = sum(p == c for p, c in answered)
    print(f"\nCATEGORY MODEL on {len(names)} item names it never saw:")
    print(f"  answered with a category: {len(answered)} ({100 * len(answered) / len(names):.0f}%), of those correct: {100 * ok / max(len(answered), 1):.0f}%")
    print(f"  said 'Other' because it was not sure: {len(names) - len(answered)} ({100 * (len(names) - len(answered)) / len(names):.0f}%)  <- the user picks these in the app")

test_on_fake(STARTER)
print()
print("HOW TO READ THIS: 'label' is what each line really is (item, total, tax...). 'correct' is how often the model agreed.")
print("The category model is allowed to say 'Other' when unsure; that is better than a confident wrong category.")

## 8. Test 2: real bills the model has never seen
First upload `Bills.zip` and let the OCR read all 25 photos (the text is cached in `ocr_cache.json`, so re-running this part is fast).

In [ ]:
BILLS_DIR = "Bills"
if not glob.glob(BILLS_DIR + "/*.jpg"):
    try:
        from google.colab import files
        print("Select Bills.zip")
        up = files.upload()
        for name, data in up.items():
            if name.lower().endswith(".zip"):
                zipfile.ZipFile(io.BytesIO(data)).extractall(".")
    except ImportError:
        pass
BILL_PHOTOS = sorted(glob.glob(BILLS_DIR + "/*.jpg"), key=lambda p: (len(p), p))
print(len(BILL_PHOTOS), "photos found" if BILL_PHOTOS else "photos found: Test 2 will be skipped, everything else still works")

In [ ]:
# docTR must really work here. If it does not, stop and show the real error instead of quietly using Tesseract.
DRY_RUN = os.environ.get("KHARCHA_DRY_RUN") == "1"          # only for testing this notebook where docTR cannot be installed
if not DRY_RUN:
    assert doctr_available(), "docTR did not install. Run the install cell again (Runtime > Restart session if needed)."
    import traceback
    try:
        _doctr()
        probe = _doctr_lines(Image.open(BILL_PHOTOS[0])) if BILL_PHOTOS else [1]
        print(f"docTR loaded and read a test photo ({len(probe)} lines): OK")
    except Exception:
        print("docTR is installed but FAILED to run. This is the real error (send it to me):")
        traceback.print_exc()
        raise SystemExit("Stopping: the results below would silently use Tesseract.")
print("OCR engine:", "docTR (Tesseract is the Nepali / rescue path)" if doctr_available() and not DRY_RUN else "Tesseract only (dry run)")

In [ ]:
OCR_BY_PHOTO = {}
if BILL_PHOTOS:
    engine_tag = "docTR" if doctr_available() and ENGINE != "tesseract" else "Tesseract"
    cache = json.load(open("ocr_cache.json", encoding="utf-8")) if os.path.exists("ocr_cache.json") else {}
    if cache.get("_engine") == engine_tag and cache.get("_n") == len(BILL_PHOTOS):
        OCR_BY_PHOTO = cache["lines"]
        print("OCR text loaded from ocr_cache.json (delete the file to read the photos again)")
    else:
        t0 = time.time()
        for p in BILL_PHOTOS:
            OCR_BY_PHOTO[os.path.basename(p)] = [l.strip() for l in ocr_lines(Image.open(p)) if l.strip()]
        json.dump({"_engine": engine_tag, "_n": len(BILL_PHOTOS), "lines": OCR_BY_PHOTO}, open("ocr_cache.json", "w", encoding="utf-8"), ensure_ascii=False)
        print(f"OCR read {len(BILL_PHOTOS)} photos in {time.time() - t0:.0f} s")
    GOLD = load_gold()
    REC = build_receipts(OCR_BY_PHOTO, GOLD)
    TRAIN_BILLS = trainable(REC)
    print(f"{len(REC)} bills labelled; {len(TRAIN_BILLS)} are used for training and testing (known total and items, not handwritten)")
    print("Lines per label:", dict(Counter(l for r in REC.values() for _, l in r["lines"])))
    print()
    print("HOW TO READ THIS: each OCR line was matched to the answer key: 'item' = its amount equals a real item price,")
    print("'total' = a total-looking line whose amount equals the real total, 'other' = everything else. Where the OCR garbled a line the model cannot learn from it.")
else:
    print("no photos: skipped")

**The honest test.** For each bill, models trained on the *other* bills (plus the fake receipts) read it. No bill is read by a model that saw it, and duplicate photos of one paper bill always stay on the same side of the split.
`AUTO` = the app would file it without asking (confidence at or above the threshold); `REVIEW` = it goes to the review screen. **The only real failure is AUTO with something wrong.**

In [ ]:
def test_on_real(rec, gold, names, k=4, seed=0, synthetic=3000):
    results = []
    for f, held in enumerate(folds(names, gold, k, seed)):
        train_names = [n for n in names if n not in held]
        m = train_models([rec[n]["lines"] for n in train_names], bill_items(rec, train_names), synthetic, quiet=True)
        for n in held:
            pred = read_lines(rec[n]["texts"], m)
            results.append({"bill": n, "gold": gold[n], "pred": pred, "conf": pred.get("confidence") or 0.0, **judge(pred, gold[n])})
    return results

def print_table(results):
    print(f"  {'bill':9}{'conf':>5}  {'app does':9}{'total':7}{'items':6}{'date':6}{'store':6} what went wrong")
    for r in sorted(results, key=lambda r: -r["conf"]):
        auto = r["conf"] >= REVIEW_THRESHOLD
        wrong = auto and not (r["total"] and r["items"])
        d = ("ok" if r["date"] else "no") if r["date"] is not None else "-"
        print(f"  {r['bill']:9}{r['conf']:5.2f}  {'AUTO' if auto else 'REVIEW':9}{'ok' if r['total'] else 'WRONG':7}{'ok' if r['items'] else 'no':6}{d:6}{'ok' if r['merchant'] else 'no':6} "
              f"{'<<< AUTO BUT WRONG: ' if wrong else ''}{diagnose(r['pred'], r['gold'])}")

if BILL_PHOTOS:
    print(f"Training {4} times (each time leaving a quarter of the bills out), then reading the left-out bills...")
    RESULTS = test_on_real(REC, GOLD, TRAIN_BILLS)
    pct = lambda xs: f"{100 * sum(xs) / len(xs):.0f}%" if xs else "-"
    d = [r["date"] for r in RESULTS if r["date"] is not None]
    print(f"\n{len(RESULTS)} bills read by models that had NOT seen them:")
    print(f"  merchant right {pct([r['merchant'] for r in RESULTS])}   date right {pct(d)}   total right {pct([r['total'] for r in RESULTS])}   all items right {pct([r['items'] for r in RESULTS])}\n")
    print_table(RESULTS)
    print()
    print("HOW TO READ THE TABLE")
    print("  total   : right = within 0.5% of the real total.        items : every item price right and no junk items.")
    print("  date    : right = the real date (3 days of slack when it came from a Nepali date); '-' = the key has no date.")
    print("  a wrong read that says REVIEW is the safety net doing its job. 'AUTO BUT WRONG' is the real failure.")
else:
    print("skipped: no photos")

### Verdict

In [ ]:
if BILL_PHOTOS:
    good = [r for r in RESULTS if r["total"] and r["items"]]
    bad = [r for r in RESULTS if not r["total"]]
    wrong_clear = sum(r["conf"] >= REVIEW_THRESHOLD for r in bad)
    good_clear = sum(r["conf"] >= REVIEW_THRESHOLD for r in good)
    tot = sum(r["total"] for r in RESULTS) / len(RESULTS)
    checks = [
        ("No wrong total is auto-approved", wrong_clear == 0, f"{wrong_clear} of {len(bad)} wrong-total bills cleared {REVIEW_THRESHOLD}",
         "The safety check. A wrong total the app files silently is the worst outcome, so this must be zero."),
        ("Most fully correct bills are auto-approved", bool(good) and good_clear / len(good) >= 0.7, f"{good_clear} of {len(good)} cleared {REVIEW_THRESHOLD} (target 70%)",
         "If this fails the app is too cautious: users get sent to review even when the read was perfect."),
        ("Total is right on at least 75% of bills", tot >= 0.75, f"{100 * tot:.0f}% (target 75%)",
         "Overall accuracy of the total. Mostly limited by the OCR (a number it never read cannot be found)."),
    ]
    for name, ok, detail, meaning in checks:
        print(("PASS  " if ok else "FAIL  ") + name + "\n      result: " + detail + "\n      meaning: " + meaning + "\n")
    print("About 20 bills: this shows the design works, it is not a guarantee. Numbers move a few points between runs.")

## 9. Train the final models, then check the hard bills
The final models learn from every scorable bill. The hard bills below were **not** used for training: handwritten (1, 24, 21) and cut off (3, 22).
They must never be auto-approved **with something wrong**.

In [ ]:
if BILL_PHOTOS:
    FINAL = train_models([REC[n]["lines"] for n in TRAIN_BILLS], bill_items(REC, TRAIN_BILLS), 4000)
else:
    FINAL = train_models(quiet=False)
MODELS = FINAL
print()
if BILL_PHOTOS:
    HARD = {"1.jpg": "handwritten (Binayak Furniture)", "24.jpg": "same handwritten bill, second photo", "21.jpg": "handwritten Nepali rice mill bill",
            "3.jpg": "photo cut off before the total", "22.jpg": "same cut-off bill, second photo"}
    ok_all = True
    print(f"{'bill':8}{'conf':>6}  {'app does':9}{'total':>9}  note")
    for n, why in HARD.items():
        if n not in REC:
            continue
        r = read_lines(REC[n]["texts"], MODELS)
        g = GOLD[n]
        auto = (r.get("confidence") or 0) >= REVIEW_THRESHOLD and not r.get("needs_review")
        if g.get("total") is not None and g.get("items"):
            j = judge(r, g)
            bad = auto and not (j["total"] and j["items"])
            note = "read " + ("correctly" if j["total"] and j["items"] else "WRONG") + f" (key: total {g['total']:g})"
        else:
            bad = auto
            note = "no reliable total exists on this photo"
        ok_all &= not bad
        print(f"{n:8}{(r.get('confidence') or 0):>6.2f}  {'AUTO' if auto else 'REVIEW':9}{str(r.get('total')):>9}  {'BAD ' if bad else 'OK  '}{why}; {note}")
    print()
    print("RESULT:", "PASS, no hard bill is auto-approved wrong." if ok_all else "FAIL, a hard bill would be filed wrongly. Send me this table.")

## 10. Your own photo
**Custom image entry point.** Use a bill that is **not** in `Bills.zip`. Either put file paths in `MY_PHOTO_PATHS`, or leave it empty and the cell lets you upload photos (Colab).
You get a six-part explanation (what the OCR read, what the model took each line for, what was decided, whether the numbers add up, how the confidence was worked out, a checklist) and then the **complete JSON**:
`api_reply` is exactly what the app/backend receives, `debug` is the reasoning. The JSON is saved to `results/`.

`read_my_photo(path_or_image)` is the one-call version if you want the result only.

In [ ]:
def _bar(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)


def show(result, trace, name="receipt", full_json=True):
    c = trace.get("confidence", {})
    conf = result.get("confidence")
    review = result.get("needs_review")
    _bar(f"RECEIPT: {name}")
    print(f"VERDICT : {'NEEDS REVIEW (the app would send this to the review screen)' if review else 'AUTO-APPROVED (the app would file it without asking)'}")
    print(f"WHY     : {result.get('reconciliation')}")
    print(f"CONFIDENCE {conf}  (threshold {c.get('review_threshold', 0.7)}; at or above goes straight in)")

    _bar("1. WHAT THE OCR READ")
    ol = trace.get("ocr_lines")
    if ol is not None:
        print(f"Engine: {trace.get('ocr_engine')}   |   {len(ol)} text lines")
        print("If a number or word you can see on the bill is missing or wrong here, the OCR is the problem, not the models.")

    if trace.get("unreadable"):
        print("\n>>> The reader decided this photo is NOT READABLE: not one total / tax / date / bill word and fewer than 3 priced lines,")
        print(">>> so the text is noise (handwriting, or Devanagari that the Latin OCR turned into nonsense). It returns no items and sends it to review.")
    if trace.get("rejoined_lines"):
        print(f"\n{len(trace['rejoined_lines'])} item(s) had their name and their numbers on separate lines; they were joined before reading, for example:")
        a, b = trace["rejoined_lines"][0]
        print(f"   {b!r} + {a!r}")
    _bar("2. LINE BY LINE: what the trained line model took each line for")
    print("sureness = how sure the model is of its label (1.00 = certain). role = what the reader finally did with the line.\n")
    print(f"{'#':>3}  {'label':9}{'sure':>5}  {'role':36} text")
    for l in trace.get("lines", []):
        role = l["role"] if l["role"] != "ignored" else "-"
        print(f"{l['index']:>3}  {l['model_label']:9}{l['model_sureness']:>5.2f}  {role[:36]:36} {l['text'][:60]}")

    _bar("3. WHAT WAS DECIDED")
    m, d, t = trace.get("merchant", {}), trace.get("date", {}), trace.get("total", {})
    print(f"Merchant : {m.get('value')!r}" + (f"   (from line {m['line_index']})" if m.get("line_index") is not None else "   (no line was sure enough)"))
    if d.get("value"):
        src = f"line {d['from_line']}" if d.get("from_line") is not None else "the text"
        print(f"Date     : {d['value']}   (from {src})" + ("   APPROXIMATE: converted from a Bikram Sambat date, can be off by 1-2 days" if d.get("approx_from_bikram_sambat") else ""))
    else:
        print("Date     : not found" + (f"   ({d['note']})" if d.get("note") else ""))
    print(f"Total    : {t.get('value')}   source: {t.get('source')}" + (f", line {t['line_index']}" if t.get("line_index") is not None else "") + f", model sureness {t.get('model_sureness')}")
    items = result.get("items", [])
    cats = {c["item"]: c["top3"] for c in trace.get("categories", [])}
    print(f"\nItems ({len(items)} found):")
    if items:
        print(f"  {'#':>2}  {'name':34}{'qty':>4}{'price':>10}  category (top 3 guesses)")
        for i, it in enumerate(items, 1):
            top = ", ".join(f"{k} {v:.2f}" for k, v in cats.get(it["name"], {}).items())
            print(f"  {i:>2}  {it['name'][:34]:34}{it.get('qty', 1):>4}{it['price']:>10.2f}  {it.get('category')}  [{top}]")
        print("  (price = the line amount printed on the bill; 'Other' with a low top guess = the model was not sure enough, so the user chooses)")
    else:
        print("  none. Either the OCR missed the item lines or the model did not recognise them as items (see section 2).")

    _bar("4. DO THE NUMBERS ADD UP? (the main check on whether the read is right)")
    a = trace.get("arithmetic", {})
    if "items_sum" in a:
        print(f"Sum of the item amounts found : {a['items_sum']}")
        print(f"Tax / VAT / service amounts   : {a.get('tax_amounts_found') or 'none found'}")
        print(f"Discount amounts              : {a.get('discount_amounts_found') or 'none found'}")
        print(f"Cash paid / change            : {a.get('cash_found') or 'none'} / {a.get('change_found') or 'none'}")
        print(f"Printed total                 : {a.get('printed_total')}   (allowed difference {a.get('tolerance')})")
        print("RESULT: " + (a.get("matched") if a.get("matched") else "items do NOT explain the total. Either an item line was missed or misread, or the total was misread."))
        print("Tip: a stray number picked up as a tax line can make a wrong read add up by luck; check the tax list above looks right.")
    else:
        print("Not run: no total line was found, so there is nothing to add up against.")

    _bar("5. HOW THE CONFIDENCE WAS WORKED OUT")
    print(f"The line model's own sureness : total {c.get('model_total_sureness')}, items on average {c.get('model_mean_item_sureness')}  ->  {c.get('model_only')}")
    print(f"Rule applied                  : {c.get('rule_applied')}")
    print(f"Final confidence              : {c.get('final')}")

    _bar("6. HOW TO CHECK IT WORKED (compare with the real bill)")
    print("[ ] merchant name matches            [ ] date matches (month and day not swapped)")
    print(f"[ ] total = {t.get('value')}  matches the bill's final amount")
    print(f"[ ] {len(items)} items found: does the bill have that many lines?  Is each price right?")
    print("[ ] if the VERDICT is AUTO-APPROVED, everything above must be right; if it is NEEDS REVIEW, it is fine for it to be wrong.")
    print("    The only real failure is AUTO-APPROVED with something wrong.")

    if full_json:
        _bar("FULL JSON (api_reply is exactly what the app/backend receives; debug is for you)")
        print(json.dumps({"api_reply": result, "debug": {k: v for k, v in trace.items()}}, indent=2, ensure_ascii=False))

In [ ]:
MY_PHOTO_PATHS = []        # e.g. ["/content/my_bill.jpg"]. Leave empty to upload in Colab.

def read_my_photo(image, models=None, explain_it=True, name="my photo"):
    """One photo (path or PIL image) in, receipt dict out. explain_it=True also prints the full report."""
    models = models or MODELS
    if isinstance(image, (str, os.PathLike)):
        name = os.path.basename(str(image))
        image = Image.open(image)
    image.load()
    trace = {}
    result = read_photo(image, models, trace)
    if explain_it:
        show(result, trace, name)
    os.makedirs("results", exist_ok=True)
    out = os.path.join("results", "result_" + os.path.splitext(name)[0] + ".json")
    json.dump({"api_reply": result, "debug": trace}, open(out, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"\n(saved {out})")
    return result

photos = list(MY_PHOTO_PATHS)
if not photos:
    try:
        from google.colab import files
        print("Choose one or more bill photos that are NOT in Bills.zip.")
        os.makedirs("my_photos", exist_ok=True)
        for name, data in files.upload().items():
            p = os.path.join("my_photos", name)
            open(p, "wb").write(data)
            photos.append(p)
    except ImportError:
        print("Not in Colab: put paths in MY_PHOTO_PATHS and run this cell again.")
for p in photos:
    try:
        from IPython.display import display
        t = Image.open(p); t.thumbnail((450, 450)); display(t)
    except Exception:
        pass
    read_my_photo(p)

## 11. Export for the backend
**Model export point.** This writes the folder `kharcha_export/` with
* `kharcha_reader.py`: the reader as one file (settings, rules, OCR, both model classes, `KharchaReader`), copied from the `# export` cells above, so it is exactly the code you tested;
* `line_model.joblib` and `category_model.joblib`: the trained models;
* `requirements.txt` and `README.txt`.

Then it **tests the export in a fresh Python process** (no notebook state) on a photo and checks the answer is identical to the notebook's. Finally it downloads `kharcha_export.zip`.
Trained models only load with the same scikit-learn version: the requirements file pins the version used here.

In [ ]:
import importlib.util, subprocess, sys

EXPORT_DIR = "kharcha_export"

def exported_source():
    """The latest run of every '# export' cell, in order."""
    history = get_ipython().user_ns["_ih"]
    latest = {}
    for src in history:
        first = src.lstrip().split("\n", 1)[0]
        if first.startswith("# export "):
            kv = dict(p.split("=") for p in first.split()[2:])
            latest[kv["name"]] = (int(kv["order"]), src)
    parts = [s for _, s in sorted(latest.values())]
    assert len(parts) == 5, f"expected 5 exported cells (settings, parse, ocr, models, pipeline), found {sorted(latest)}: run the cells above first"
    body = "\n\n\n".join(parts).replace('print("settings loaded")\n', "")
    return "# kharcha_reader.py: generated by the Kharcha notebook (part 11). Do not edit by hand; change the notebook and export again.\n" + body + "\n"

shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR)
open(os.path.join(EXPORT_DIR, "kharcha_reader.py"), "w", encoding="utf-8").write(exported_source())

# Load the exported module and rebuild the trained models as ITS classes, so the saved files do not depend on the notebook.
spec = importlib.util.spec_from_file_location("kharcha_reader", os.path.join(EXPORT_DIR, "kharcha_reader.py"))
kr = importlib.util.module_from_spec(spec)
sys.modules["kharcha_reader"] = kr
spec.loader.exec_module(kr)
exported = kr.Models(kr.LineModel(), kr.CategoryModel())
exported.line.__dict__.update(MODELS.line.__dict__)
exported.category.__dict__.update(MODELS.category.__dict__)
exported.save(EXPORT_DIR)

import sklearn
open(os.path.join(EXPORT_DIR, "requirements.txt"), "w").write(
    f"scikit-learn=={sklearn.__version__}\njoblib\nscipy\nnumpy\npillow\npytesseract\nopencv-python-headless\npython-doctr[torch]\n")
open(os.path.join(EXPORT_DIR, "README.txt"), "w").write(
"""Kharcha reader export
=====================
System:   apt install tesseract-ocr tesseract-ocr-eng tesseract-ocr-nep      (Windows: install Tesseract, set TESSERACT_CMD)
Python:   pip install -r requirements.txt        (scikit-learn must be the pinned version, or retrain in the notebook)

Use it:
    from kharcha_reader import KharchaReader
    reader = KharchaReader("path/to/this/folder")      # raises ModelsMissing if the .joblib files are gone
    result = reader.read("bill.jpg")                   # path or PIL image
    # result = {merchant?, date?, total, items:[{name, qty, price, category}], confidence, needs_review, reconciliation}
    # needs_review True -> send it to the review screen; False -> file it.
    # readable False (only on noise) -> nothing could be read; ask the user to type it in.

Tiny HTTP wrapper (FastAPI):
    from fastapi import FastAPI, Request
    import io; from PIL import Image
    app = FastAPI(); reader = KharchaReader(".")
    @app.post("/read")
    async def read(req: Request): return reader.read(Image.open(io.BytesIO(await req.body())))
""")
print("written:", sorted(os.listdir(EXPORT_DIR)))

In [ ]:
# Test the export the way the backend will use it: a fresh Python process that only sees the export folder.
probe_photo = (BILL_PHOTOS[7] if BILL_PHOTOS else None)
if probe_photo is None:
    probe_photo = render(["CORNER MART", "Date: 15/10/2026", "Rice 5kg   640.00", "Milk 1l   140.00", "Grand Total   780.00"], "probe.png") and "probe.png"
child = (
    "import json, sys\n"
    "from kharcha_reader import KharchaReader\n"
    "r = KharchaReader('.')\n"
    "print(json.dumps(r.read(sys.argv[1]), ensure_ascii=False, sort_keys=True))\n")
p = subprocess.run([sys.executable, "-c", child, os.path.abspath(probe_photo)], cwd=EXPORT_DIR, capture_output=True, text=True)
if p.returncode != 0:
    print("EXPORT TEST FAILED, the error is:\n", p.stderr[-1500:])
else:
    from_export = json.loads(p.stdout.strip().splitlines()[-1])
    from_notebook = json.loads(json.dumps(read_photo(probe_photo, MODELS), sort_keys=True))
    print("photo used:", probe_photo)
    print("export result: merchant =", from_export.get("merchant"), "| total =", from_export.get("total"), "| items =", len(from_export["items"]),
          "| confidence =", from_export.get("confidence"), "| needs_review =", from_export.get("needs_review"))
    print("RESULT:", "PASS, the exported reader gives exactly the same answer as the notebook." if from_export == from_notebook
          else "DIFFERENT from the notebook: send me both outputs.")
shutil.make_archive("kharcha_export", "zip", EXPORT_DIR)
try:
    from google.colab import files
    files.download("kharcha_export.zip")
except ImportError:
    print("kharcha_export.zip written (not in Colab, so not downloaded)")